# 读取配置信息

In [1]:
# ============================================================================
#  读取项目配置 config.yaml，并推导所有中间产物文件名
#  本 cell 自包含：先定义 WORK_DIR 并 chdir（Run All 计时起点也在此），后续 cell 直接用这里暴露的变量。
#  注：若你另有「定义 WORK_DIR 的第一个 cell」，此处会以本 cell 为准覆盖它，那个 cell 可删掉。
# ============================================================================
import yaml, os, time, unicodedata
from pathlib import Path
from collections import OrderedDict

# ---------- 工作目录（须在读取 config.yaml 之前 chdir，使后续所有相对路径都基于此）----------
# WORK_DIR = Path(r"Z:\unblinded\shr-a1811-306\shr-a1811-iii-306-cn\99_archive\define demo v1")
WORK_DIR = Path(r"C:\Users\chenqx\Desktop\ai-define-master\ai-define-master")
assert WORK_DIR.exists(), f"WORK_DIR 不存在: {WORK_DIR}"
os.chdir(WORK_DIR)
_notebook_start = time.time()   # ← Run All 计时起点（build_define 末尾据此算总耗时）

CFG = yaml.safe_load(Path("config.yaml").read_text(encoding="utf-8")) or {}

# ---------- 语言 ----------
LANG     = str(CFG.get("language", "CN")).upper()
assert LANG in ("CN", "EN"), f"config.language 必须是 CN 或 EN，当前: {LANG!r}"
XML_LANG = {"CN": "zh", "EN": "en"}[LANG]   # define.xml TranslatedText 的 xml:lang
SUF      = {"CN": "_cn", "EN": "_en"}[LANG]  # 取 name/description/decode 时的字段后缀

# ---------- 目录 ----------
SDTM_DIR     = CFG["sdtm_dir"]                          # xpt + acrf.pdf 所在目录
DATASETS_DIR = CFG["output_files"]["datasets_dir"]       # 各域 yaml 输出目录
DEPT_DIR     = Path(CFG.get("dept_dir",  "dept_std"))    # 部门标准库 xlsx/yaml 所在目录
STUDY_DIR    = Path(CFG.get("study_dir", "study_std"))   # 项目层提取产物目录
DEPT_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 部门级文件（输入，完整路径） ----------
_D = CFG["dept_files"]
SDTM_CT_XLSX    = DEPT_DIR / _D["sdtm_ct_xlsx"]
METHODS_XLSX    = DEPT_DIR / _D["methods_xlsx"]
VALUELEVEL_XLSX = DEPT_DIR / _D["valuelevel_xlsx"]
COMMENTS_XLSX   = DEPT_DIR / _D["comments_xlsx"]

# ---------- 项目专属文件（输入，统一从 STUDY_DIR = study_std\ 读取） ----------
_S = CFG["study_files"]
SPEC_XLSX  = STUDY_DIR / _S["spec_xlsx"]                 # 若 spec 仍放工作目录根: 改回 _S["spec_xlsx"]
STDCL_XLSX = STUDY_DIR / _S["stdcl_xlsx"]
ACRF_PDF   = str(Path(SDTM_DIR) / _S["acrf_pdf"])       # acrf.pdf 和 xpt 同目录

# 研究号: 优先读 config.yaml 的 study_id，否则从 spec 文件名推导
# spec-sdtm-SHR-1234-III-301-define.xlsx → SHR-1234-III-301
_spec_stem = Path(SPEC_XLSX).stem                        # spec-sdtm-SHR-1234-III-301-define
_study_id_inferred = _spec_stem.removeprefix("spec-sdtm-").removesuffix("-define")
STUDY_ID = CFG.get("study_id") or _study_id_inferred
_sid_src = "来自 config.yaml" if CFG.get("study_id") else "从 spec 文件名推导"

# ---------- 部门级中间产物（输出到 DEPT_DIR） ----------
SDTM_CT_YAML      = DEPT_DIR / (Path(_D["sdtm_ct_xlsx"]).stem + ".yaml")
METHODS_YAML      = DEPT_DIR / (Path(_D["methods_xlsx"]).stem + ".yaml")
VALUELEVEL_YAML   = DEPT_DIR / (Path(_D["valuelevel_xlsx"]).stem + ".yaml")
WHERECLAUSE_YAML  = DEPT_DIR / ("whereclause-" + Path(_D["valuelevel_xlsx"]).stem.split("valuelevel-", 1)[1] + ".yaml")
COMMENTS_LIB_YAML = DEPT_DIR / (Path(_D["comments_xlsx"]).stem + ".yaml")

# ---------- 项目层中间产物（输出到 STUDY_DIR） ----------
ACRF_PAGES_YAML   = STUDY_DIR / (Path(_S["acrf_pdf"]).stem + ".yaml")   # acrf_pages.yaml
STDCL_YAML        = STUDY_DIR / "standard_codelist.yaml"
TOC_YAML          = Path(DATASETS_DIR) / "toc.yaml"     # build_domain_yaml 写出，在 datasets/ 下

# ---------- 项目层聚合产物（输出到 STUDY_DIR） ----------
CODELISTS_YAML    = STUDY_DIR / "codelists.yaml"
OUT_METHODS_YAML  = STUDY_DIR / "methods.yaml"
OUT_COMMENTS_YAML = STUDY_DIR / "comments.yaml"

# ---------- 最终产物（工作目录） ----------
DEFINE_XML = CFG["output_files"]["define_xml"]

# ============================================================================
#  公共工具（注册一次，后续所有 cell 直接复用；须先运行本 config cell）
#  · OrderedDict / Flow 的 YAML representer —— 全 kernel 生效，各 cell 不必再注册
#  · cidx / cell / make_getter —— 统一表头取值助手（表头键 strip，更稳健）
# ============================================================================
yaml.add_representer(OrderedDict, lambda d,x: d.represent_mapping("tag:yaml.org,2002:map", x.items()))
class Flow(dict): pass
yaml.add_representer(Flow, lambda d,x: d.represent_mapping("tag:yaml.org,2002:map", x.items(), flow_style=True))

def cidx(hdr):
    return {(c.strip() if isinstance(c,str) else c): i for i,c in enumerate(hdr) if c is not None}
def cell(row, idx, col):
    if row is None: return None
    i=idx.get(col); v=row[i] if (i is not None and i<len(row)) else None
    return None if (v is None or str(v).strip()=="") else v
def make_getter(hdr):
    idx=cidx(hdr)
    return idx, (lambda r,c: cell(r, idx, c))

# ---------- 打印确认 ----------
def _w(s):      return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)
def _pad(s, n): return s + " " * max(0, n - _w(s))
def _grp(title, files):                              # 目录标题 + 逐文件 bullet
    print(f"     📁 {title}")
    for f in files: print(f"          · {Path(f).name}")
_BAR, _SUB, _IND = "═"*58, "─"*58, " "*15

print("\n" + _BAR)
print("  📦  Define-XML 构建配置")
print(_BAR)
print(f"  📁 {_pad('工作目录',8)}  {WORK_DIR}")
print(f"  🌐 {_pad('语言',8)}  {LANG}   ·   xml:lang={XML_LANG}   ·   字段后缀={SUF}")
print(f"  🔖 {_pad('研究号',8)}  {STUDY_ID}   ({_sid_src})")
print(f"  📂 {_pad('目录',8)}  sdtm={SDTM_DIR}")
print(f"{_IND}dept={DEPT_DIR}{os.sep}   study={STUDY_DIR}{os.sep}   datasets={DATASETS_DIR}{os.sep}")
print(_SUB)
print("  📥 输入")
print(f"     📄 spec    {SPEC_XLSX}")
print(f"     📄 stdcl   {STDCL_XLSX}")
print(f"     📄 aCRF    {ACRF_PDF}")
_grp(f"{DEPT_DIR}{os.sep}  (部门标准库)", [SDTM_CT_XLSX, METHODS_XLSX, VALUELEVEL_XLSX, COMMENTS_XLSX])
print(_SUB)
print("  ⚙️  中间产物")
_grp(f"{DEPT_DIR}{os.sep}",  [SDTM_CT_YAML, METHODS_YAML, VALUELEVEL_YAML, WHERECLAUSE_YAML, COMMENTS_LIB_YAML])
_grp(f"{STUDY_DIR}{os.sep}", [ACRF_PAGES_YAML, STDCL_YAML, CODELISTS_YAML, OUT_METHODS_YAML, OUT_COMMENTS_YAML])
_grp(f"{DATASETS_DIR}{os.sep}", [TOC_YAML])
print(_SUB)
print(f"  📤 {_pad('最终产物',8)}  {DEFINE_XML}")
print(_BAR)


══════════════════════════════════════════════════════════
  📦  Define-XML 构建配置
══════════════════════════════════════════════════════════
  📁 工作目录  C:\Users\chenqx\Desktop\ai-define-master\ai-define-master
  🌐 语言      CN   ·   xml:lang=zh   ·   字段后缀=_cn
  🔖 研究号    SHR-1234-III-301   (来自 config.yaml)
  📂 目录      sdtm=m5\datasets\tabulations\sdtm
               dept=dept_std\   study=study_std\   datasets=datasets\
──────────────────────────────────────────────────────────
  📥 输入
     📄 spec    study_std\spec-sdtm-SHR-1234-III-301-define.xlsx
     📄 stdcl   study_std\standard-codelist-SHR-1234-III-301_for_define.xlsx
     📄 aCRF    m5\datasets\tabulations\sdtm\acrf.pdf
     📁 dept_std\  (部门标准库)
          · SDTM_CT_2024-09-27.xlsx
          · computation-methods-studyname_sdtm_v1.1.xlsx
          · valuelevel-studyname_sdtm_v1.1.xlsx
          · comments-studyname_sdtm_v1.1.xlsx
──────────────────────────────────────────────────────────
  ⚙️  中间产物
     📁 dept_std\
          · SDTM_CT_20

# 读取部门/行业标准文件

## CDISC CT

In [2]:
# ============================================================================
#  build_ct_yaml —— 把 CDISC SDTM CT 包(xlsx)提取成可按 (码表短名, 值) 查 NCI 码的 yaml
#  结构: {短名: {nci_code, extensible, name, items: {提交值: 术语NCI码}}}
# ============================================================================
import openpyxl, yaml, os, time
from collections import OrderedDict
from pathlib import Path

SRC = SDTM_CT_XLSX    # config cell: DEPT_DIR / dept_files.sdtm_ct_xlsx
OUT = SDTM_CT_YAML    # config cell: DEPT_DIR / (stem + ".yaml")
_t0 = time.time()

wb=openpyxl.load_workbook(SRC, read_only=True, data_only=True)
ws=wb["Terminology"]
it=ws.iter_rows(values_only=True); next(it)            # 跳表头
# 列: 0 Code 1 CodelistCode 2 Extensible 3 Name 4 SubVal 5 Synonyms 6 Def 7 PrefTerm 8 Std

by_cc=OrderedDict()                                    # 码表 C-code -> dict
order=[]
n_term=0
n_orphan=0                                             # 术语先于码表头出现(容错)的码表数
for r in it:
    code, clcode, ext, name, subval = r[0], r[1], r[2], r[3], r[4]
    if not code: continue
    if clcode in (None, ""):                           # 码表头行
        by_cc[code]=OrderedDict(nci_code=code, submission_value=subval, name=name,
                                extensible=(str(ext).strip().lower()=="yes"), items=OrderedDict())
        order.append(code)
    else:                                              # 术语行
        cl=by_cc.get(clcode)
        if cl is None:                                 # 术语先于头出现(容错)
            cl=OrderedDict(nci_code=clcode, submission_value=None, name=name, extensible=None, items=OrderedDict())
            by_cc[clcode]=cl; order.append(clcode); n_orphan+=1
        if subval is not None:
            cl["items"][str(subval)]=code              # 提交值 -> 术语 NCI 码
            n_term+=1

# 按短名(submission_value)输出；短名缺失或冲突则退回 C-code
out=OrderedDict(); seen={}; n_collision=0
for cc in order:
    cl=by_cc[cc]
    key=cl["submission_value"] or cc
    if key in seen:                                    # 短名冲突(罕见)→加 C-code 后缀
        key=f"{key}#{cc}"; n_collision+=1
    seen[key]=cc
    out[key]=OrderedDict(nci_code=cl["nci_code"], extensible=cl["extensible"],
                         name=cl["name"], items=cl["items"])

with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# CDISC SDTM CT 提取自 {SRC}。结构: 短名 -> {{nci_code, extensible, name, items:{{提交值:术语NCI码}}}}\n")
    yaml.dump(out, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=200)

# ---------- 统计 ----------
n_cl       = len(out)
n_ext      = sum(1 for v in out.values() if v["extensible"] is True)
n_nonext   = sum(1 for v in out.values() if v["extensible"] is False)
n_unknown  = sum(1 for v in out.values() if v["extensible"] is None)
n_empty    = sum(1 for v in out.values() if not v["items"])
n_fallback = sum(1 for cc in order if not by_cc[cc]["submission_value"])   # 无短名→退回 C-code
_top       = sorted(((k, len(v["items"])) for k,v in out.items()), key=lambda kv: kv[1], reverse=True)[:5]

# ---------- 打印确认（_pad 来自 config cell；_BAR/_SUB/_sz 本地） ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  📚  CDISC SDTM CT 提取")
print(_BAR)
print(f"  📥 {_pad('输入',6)}{SRC}   ({_sz(SRC)})")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s")
print(_SUB)
_avg = n_term/n_cl if n_cl else 0
print(f"  📊 码表 {n_cl} 个   ·   术语 {n_term} 个   ·   平均 {_avg:.0f} 项/码表")
print(f"       可扩展 extensible  {n_ext:>5}")
print(f"       不可扩展           {n_nonext:>5}")
if n_unknown: print(f"       扩展性未知(容错)   {n_unknown:>5}")
if n_empty:   print(f"       空码表(0 项)       {n_empty:>5}")
print(f"  🔑 命名")
print(f"       用短名 submission   {n_cl-n_fallback:>5}")
if n_fallback:  print(f"       回退 C-code(无短名) {n_fallback:>5}")
if n_collision: print(f"       短名冲突(+#Ccode)  {n_collision:>5}")
print(f"  🏆 最大码表 (按术语数)")
for nm,c in _top: print(f"       · {_pad(nm,14)}{c:>4}")
print(_BAR)


══════════════════════════════════════════════════════════
  📚  CDISC SDTM CT 提取
══════════════════════════════════════════════════════════
  📥 输入  dept_std\SDTM_CT_2024-09-27.xlsx   (3.3 MB)
  📤 输出  dept_std\SDTM_CT_2024-09-27.yaml   (1.5 MB)
  ⏱  耗时  5.4 s
──────────────────────────────────────────────────────────
  📊 码表 1145 个   ·   术语 42831 个   ·   平均 37 项/码表
       可扩展 extensible    266
       不可扩展             879
  🔑 命名
       用短名 submission    1145
  🏆 最大码表 (按术语数)
       · LBTESTCD      2447
       · LBTEST        2447
       · MICROORG      1675
       · LOC           1391
       · UNIT           922
══════════════════════════════════════════════════════════


## Methods

In [3]:
# ============================================================================
#  提取部门级 computation methods → 与源文件同名的 .yaml (全量, 不过滤)
#  key = MethodOID 去掉 'MT.' 前缀(与 domain yaml 的 method 裸名引用对齐, 如 method: USUBJID)
#  项目筛选留给下游: 按每个 datasets/*.yaml 里出现的 method 引用取子集
#  依赖: openpyxl pyyaml
# ============================================================================
import yaml, openpyxl, os, time
from pathlib import Path
from collections import OrderedDict, Counter

# ---------- 配置 ----------
SRC        = METHODS_XLSX     # config cell: DEPT_DIR / dept_files.methods_xlsx
SHEET      = "Methods"
OID_PREFIX = "MT."
OUT        = METHODS_YAML     # config cell: DEPT_DIR / (stem + ".yaml")

# 源列 -> 输出字段(顺序即 yaml 内字段顺序; 仅在有值时写入)
FIELDS = [
    ("MethodName",   "name_en"),        ("MethodNameCN", "name_cn"),
    ("MethodType",   "type"),
    ("Description",  "description_en"),  ("DescriptionCN","description_cn"),
    ("DocsleafID",   "doc_id"),          ("PageRefType",  "page_ref_type"),
    ("PageRefs",     "page_refs"),       # 文档/页码引用：部门版常空, 项目版才填
]
_t0 = time.time()

wb=openpyxl.load_workbook(SRC, read_only=True, data_only=True)
ws=wb[SHEET]; it=ws.iter_rows(values_only=True); hdr=next(it)
H, g = make_getter(hdr)

out=OrderedDict(); dups=[]; skipped=0
for r in it:                                                # it 已 next 过表头
    oid=g(r,"MethodOID")
    if not oid:                                             # 空行跳过
        skipped+=1; continue
    key=str(oid)[len(OID_PREFIX):] if str(oid).startswith(OID_PREFIX) else str(oid)
    if key in out:                                          # 重复 OID: 保留首次, 记录
        dups.append(key); continue
    m=OrderedDict()
    for col, ok in FIELDS:                                  # 单次取值, 数据驱动
        v=g(r,col)
        if v: m[ok]=v
    out[key]=m

out=OrderedDict(sorted(out.items()))
with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# 部门级 computation methods 全量提取自 {SRC}(key = MethodOID 去 '{OID_PREFIX}')。项目层面按 datasets/*.yaml 的 method 引用筛选。\n")
    yaml.dump(out, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=200)

# ---------- 统计 ----------
n        = len(out)
by_type  = Counter((m.get("type") or "（未填）") for m in out.values())
has_cn   = sum(1 for m in out.values() if m.get("name_cn"))
has_desc = sum(1 for m in out.values() if m.get("description_en") or m.get("description_cn"))
has_doc  = sum(1 for m in out.values() if m.get("doc_id") or m.get("page_ref_type") or m.get("page_refs"))
no_name  = sum(1 for m in out.values() if not (m.get("name_en") or m.get("name_cn")))

# ---------- 打印确认（_pad 来自 config cell；_BAR/_SUB/_sz 本地） ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  🧮  部门级 Computation Methods 提取")
print(_BAR)
print(f"  📥 {_pad('输入',6)}{SRC}   ({_sz(SRC)})")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s   ·   跳过空行 {skipped}")
print(_SUB)
print(f"  📊 方法 {n} 个")
print(f"  🏷 按类型 (MethodType)")
for t,c in by_type.most_common(): print(f"       · {_pad(str(t),14)}{c:>4}")
print(f"  🌐 内容覆盖")
print(f"       · 有中文名 name_cn      {has_cn:>4} / {n}")
print(f"       · 有描述 description    {has_desc:>4} / {n}")
print(f"       · 含文档/页码引用        {has_doc:>4} / {n}   {'(部门版通常为空)' if has_doc==0 else ''}")
if no_name: print(f"       · ⚠ 无任何名称           {no_name:>4}")
print(_SUB)
print("  💡 key = MethodOID 去 'MT.' 前缀；项目层按 datasets/*.yaml 的 method 引用筛子集")
print(_BAR)
if dups: print(f"  ⚠ [重复OID] 保留首次, 请核源文件: {sorted(set(dups))}")


══════════════════════════════════════════════════════════
  🧮  部门级 Computation Methods 提取
══════════════════════════════════════════════════════════
  📥 输入  dept_std\computation-methods-studyname_sdtm_v1.1.xlsx   (29.7 KB)
  📤 输出  dept_std\computation-methods-studyname_sdtm_v1.1.yaml   (14.4 KB)
  ⏱  耗时  0.0 s   ·   跳过空行 1
──────────────────────────────────────────────────────────
  📊 方法 49 个
  🏷 按类型 (MethodType)
       · Computation     49
  🌐 内容覆盖
       · 有中文名 name_cn        49 / 49
       · 有描述 description      49 / 49
       · 含文档/页码引用           0 / 49   (部门版通常为空)
──────────────────────────────────────────────────────────
  💡 key = MethodOID 去 'MT.' 前缀；项目层按 datasets/*.yaml 的 method 引用筛子集
══════════════════════════════════════════════════════════
  ⚠ [重复OID] 保留首次, 请核源文件: ['SVUPDES']


## Value Level Metadata

In [4]:
# ============================================================================
#  提取 VLM sheet → valuelevel-studyname_sdtm_v1.1.yaml
#  key = VLMOID (VL.数据集.变量); 值 = 值项列表(按 OrderNumber 排序)。
#  method 保留原始 MethodOID(含 MT.); description 双语; 部门级全量。
#  跳过 QVAL 行(SUPP QVAL 值级来自实际数据, 由 build_domain 生成, 不在标准文件)。
#  where → whereclause-*.yaml 的 key。依赖: openpyxl pyyaml
# ============================================================================
import yaml, openpyxl, os, time
from collections import OrderedDict, defaultdict, Counter

SRC   = VALUELEVEL_XLSX    # config cell: DEPT_DIR / dept_files.valuelevel_xlsx
SHEET = "VLM"
OUT   = VALUELEVEL_YAML    # config cell: DEPT_DIR / (stem + ".yaml")

# 源列 -> 输出键 + 取值方式(顺序即 yaml 字段顺序)
#   str   = 有值才写;  num = 非None才写并转数字;  bool = 恒写布尔;  origin = 嵌套(type[+source])
FIELDS = [
    ("OrderNumber",       "order",              "num"),
    ("Mandatory",         "mandatory",          "bool"),
    ("WhereClauseOID",    "where",              "str"),
    ("VariableName",      "variable",           "str"),
    ("XMLDataType",       "data_type",          "str"),
    (None,                "origin",             "origin"),
    ("CodeListName",      "codelist",           "str"),
    ("MethodOID",         "method",             "str"),   # 原样保留 MT.
    ("Description",       "description_en",     "str"),
    ("DescriptionCN",     "description_cn",     "str"),
    ("SignificantDigits", "significant_digits", "num"),
    ("DisplayFormat",     "display_format",     "str"),
    ("Length",            "length",             "num"),
    ("CRF Page",          "crf_page",           "str"),
    ("Study Derivation",  "study_derivation",   "str"),
]
_t0 = time.time()

wb=openpyxl.load_workbook(SRC, read_only=True, data_only=True)
ws=wb[SHEET]; it=ws.iter_rows(values_only=True); hdr=next(it)
H, g = make_getter(hdr)
def num(v):
    try: return int(v)
    except (TypeError,ValueError): return v

groups=defaultdict(list); dups=defaultdict(int); skip_qval=0
for r in it:
    vlmoid=g(r,"VLMOID"); itemoid=g(r,"ItemOID")
    if not vlmoid or not itemoid: continue
    if g(r,"VariableName")=="QVAL":                            # SUPP QVAL 值级 → 实际数据生成, 跳过
        skip_qval+=1; continue
    core=str(vlmoid)[3:]                                        # VL.SE.SEENDTC -> SE.SEENDTC
    pref="IT."+core+"."
    item=str(itemoid)[len(pref):] if str(itemoid).startswith(pref) else str(itemoid)

    e=OrderedDict(); e["item"]=item
    for col, ok, kind in FIELDS:                               # 数据驱动, 单次取值
        if kind=="str":
            v=g(r,col)
            if v: e[ok]=v
        elif kind=="num":
            v=g(r,col)
            if v is not None: e[ok]=num(v)
        elif kind=="bool":
            e[ok]=(str(g(r,col)).lower()=="yes")
        elif kind=="origin":
            ot=g(r,"Origin")
            if ot:
                o=OrderedDict(); o["type"]=ot
                src=g(r,"Source")
                if src: o["source"]=src
                e[ok]=o

    dups[(vlmoid,item)]+=1
    groups[vlmoid].append(e)

out=OrderedDict()
for vlmoid in sorted(groups):
    out[vlmoid]=sorted(groups[vlmoid], key=lambda x: (x.get("order", 1e9), x["item"]))

with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# 值级元数据(VLM)提取自 {SRC}(key=VLMOID, 值=值项列表; method 含 MT. 原样; description 双语; 已剔除 QVAL)。where → whereclause-*.yaml。\n")
    yaml.dump(out,f,allow_unicode=True,default_flow_style=False,sort_keys=False,width=200)

# ---------- 统计 ----------
n_vlm   = len(out)
allit   = [e for v in out.values() for e in v]
nitems  = len(allit)
avg     = nitems/n_vlm if n_vlm else 0
with_where  = sum(1 for e in allit if "where"    in e)
with_method = sum(1 for e in allit if "method"   in e)
with_cl     = sum(1 for e in allit if "codelist" in e)
with_origin = sum(1 for e in allit if "origin"   in e)
origin_dist = Counter(e["origin"]["type"] for e in allit if "origin" in e)
_top        = sorted(((k,len(v)) for k,v in out.items()), key=lambda kv: kv[1], reverse=True)[:5]
dup_pairs   = [k for k,c in dups.items() if c>1]

# ---------- 打印确认（_pad 来自 config cell；_BAR/_SUB/_sz 本地） ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  🧬  值级元数据 (VLM) 提取")
print(_BAR)
print(f"  📥 {_pad('输入',6)}{SRC}   ({_sz(SRC)})")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s   ·   跳过 QVAL 行 {skip_qval}")
print(_SUB)
print(f"  📊 VLMOID {n_vlm} 个   ·   值项 {nitems} 个   ·   平均 {avg:.1f} 项/VLMOID")
print(f"  🔎 值项覆盖")
print(f"       · 带 where 条件   {with_where:>4} / {nitems}")
print(f"       · 带 method      {with_method:>4} / {nitems}")
print(f"       · 带 codelist    {with_cl:>4} / {nitems}")
print(f"       · 带 origin      {with_origin:>4} / {nitems}")
print(f"  🌱 Origin 类型分布")
for t,c in origin_dist.most_common(): print(f"       · {_pad(str(t),12)}{c:>4}")
print(f"  🏆 最大 VLMOID (按值项数)")
for k,c in _top: print(f"       · {_pad(k,18)}{c:>3}")
print(_SUB)
print("  💡 key=VLMOID(VL.数据集.变量)；method 含 MT. 原样；where → whereclause-*.yaml；QVAL 已剔除")
print(_BAR)
if dup_pairs: print(f"  ⚠ [多 origin] 同 VLMOID+item 多行(已保留, 请核): {[f'{a}.{b}' for a,b in dup_pairs]}")


══════════════════════════════════════════════════════════


  🧬  值级元数据 (VLM) 提取
══════════════════════════════════════════════════════════
  📥 输入  dept_std\valuelevel-studyname_sdtm_v1.1.xlsx   (38.4 KB)
  📤 输出  dept_std\valuelevel-studyname_sdtm_v1.1.yaml   (23.0 KB)
  ⏱  耗时  0.1 s   ·   跳过 QVAL 行 23
──────────────────────────────────────────────────────────
  📊 VLMOID 16 个   ·   值项 86 个   ·   平均 5.4 项/VLMOID
  🔎 值项覆盖
       · 带 where 条件     86 / 86
       · 带 method        17 / 86
       · 带 codelist      31 / 86
       · 带 origin        86 / 86
  🌱 Origin 类型分布
       · Protocol      51
       · Derived       17
       · Collected     12
       · Assigned       6
  🏆 最大 VLMOID (按值项数)
       · VL.TS.TSVAL        51
       · VL.EG.EGSTRESC      3
       · VL.RS.RSORRES       3
       · VL.RS.RSSTRESC      3
       · VL.SE.SEENDTC       3
──────────────────────────────────────────────────────────
  💡 key=VLMOID(VL.数据集.变量)；method 含 MT. 原样；where → whereclause-*.yaml；QVAL 已剔除
══════════════════════════════════════════════════════════


## Where Clause

In [5]:
# ============================================================================
#  提取 WhereClause sheet → whereclause-studyname_sdtm_v1.1.yaml
#  key = WhereClauseOID; 值 = 条件列表(每个 RangeCheck 一条, 多条件即 AND)。
#  同一 (variable, comparator) 多行(IN 多值)在单条件内折叠为 values。
#  跳过 QNAM 行(SUPP QVAL/QNAM 值级来自实际数据, 由 build_domain 生成, 不在标准文件)。
#  部门级全量, 不过滤研究。依赖: openpyxl pyyaml
# ============================================================================
import yaml, openpyxl, os, time
from collections import OrderedDict, Counter

SRC   = VALUELEVEL_XLSX     # config cell: DEPT_DIR / dept_files.valuelevel_xlsx
SHEET = "WhereClause"
OUT   = WHERECLAUSE_YAML    # config cell: DEPT_DIR / ("whereclause-" + stem后缀 + ".yaml")
_t0 = time.time()

wb=openpyxl.load_workbook(SRC, read_only=True, data_only=True)
ws=wb[SHEET]; it=ws.iter_rows(values_only=True); hdr=next(it)
H, g = make_getter(hdr)

out=OrderedDict(); skip_qnam=0
for r in it:
    oid=g(r,"WhereClauseOID")
    if not oid: continue
    var=g(r,"VariableName"); comp=g(r,"Comparator"); cv=g(r,"CheckValue"); com=g(r,"CommentOID")
    if var=="QNAM":                                            # SUPP QNAM 值级 → 实际数据生成, 跳过
        skip_qnam+=1; continue
    conds=out.setdefault(oid, [])
    cond=next((c for c in conds if c["variable"]==var and c["comparator"]==comp), None)
    if cond is None:
        cond=OrderedDict(); cond["dataset"]=g(r,"Dataset"); cond["variable"]=var
        cond["comparator"]=comp; cond["values"]=[]
        if com: cond["comment"]=com
        conds.append(cond)
    if cv is not None and cv not in cond["values"]:
        cond["values"].append(cv)

out=OrderedDict(sorted(out.items()))
with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# WhereClause 提取自 {SRC}(key=WhereClauseOID, 值=条件列表; 多条件即 AND, IN 多值折叠为 values; 已剔除 QNAM)。\n")
    yaml.dump(out,f,allow_unicode=True,default_flow_style=False,sort_keys=False,width=200)

# ---------- 统计 ----------
n_wc      = len(out)
allc      = [c for v in out.values() for c in v]
n_cond    = len(allc)
multi     = sum(1 for v in out.values() if len(v)>1)              # 多条件 AND
comp_dist = Counter((c.get("comparator") or "（未填）") for c in allc)
ds_dist   = Counter((c.get("dataset")    or "（未填）") for c in allc)
n_in      = sum(1 for c in allc if len(c["values"])>1)            # IN 多值
n_comment = sum(1 for c in allc if "comment" in c)
n_empty   = sum(1 for c in allc if not c["values"])              # 异常: 无 CheckValue
_top      = sorted(out.items(), key=lambda kv:(len(kv[1]), sum(len(c["values"]) for c in kv[1])), reverse=True)[:5]

# ---------- 打印确认（_pad 来自 config cell；_BAR/_SUB/_sz 本地） ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  🔗  WhereClause 条件提取")
print(_BAR)
print(f"  📥 {_pad('输入',6)}{SRC}   ({_sz(SRC)})")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s   ·   跳过 QNAM 行 {skip_qnam}")
print(_SUB)
print(f"  📊 where {n_wc} 条   ·   条件 {n_cond} 个   ·   多条件 AND {multi} 个")
print(f"  🔣 比较符 (Comparator)")
for t,c in comp_dist.most_common(): print(f"       · {_pad(str(t),8)}{c:>4}")
print(f"  🗂 按数据集 (top 5)")
for t,c in ds_dist.most_common(5): print(f"       · {_pad(str(t),8)}{c:>4}")
print(f"  🔎 条件覆盖")
print(f"       · IN 多值 (>1)   {n_in:>4} / {n_cond}")
print(f"       · 带 comment     {n_comment:>4} / {n_cond}")
if n_empty: print(f"       · ⚠ 空值条件      {n_empty:>4}")
print(f"  🏆 最复杂 where (按条件数)")
for k,v in _top: print(f"       · {_pad(k,16)}{len(v)} 条件 / {sum(len(c['values']) for c in v)} 值")
print(_SUB)
print("  💡 key=WhereClauseOID；多条件即 AND；IN 多值折叠为 values；QNAM 已剔除")
print(_BAR)


══════════════════════════════════════════════════════════
  🔗  WhereClause 条件提取
══════════════════════════════════════════════════════════
  📥 输入  dept_std\valuelevel-studyname_sdtm_v1.1.xlsx   (38.4 KB)
  📤 输出  dept_std\whereclause-studyname_sdtm_v1.1.yaml   (9.5 KB)
  ⏱  耗时  0.1 s   ·   跳过 QNAM 行 23
──────────────────────────────────────────────────────────
  📊 where 83 条   ·   条件 90 个   ·   多条件 AND 7 个
  🔣 比较符 (Comparator)
       · EQ        79
       · IN        11
  🗂 按数据集 (top 5)
       · TS        51
       · RS        14
       · EG        10
       · RP         6
       · APRP       4
  🔎 条件覆盖
       · IN 多值 (>1)      8 / 90
       · 带 comment        0 / 90
  🏆 最复杂 where (按条件数)
       · WC.EG.EGORRES.ECG32 条件 / 4 值
       · WC.EG.EGSTRESC.ECG32 条件 / 4 值
       · WC.EG.EGORRES.ECG12 条件 / 2 值
       · WC.EG.EGSTRESC.ECG12 条件 / 2 值
       · WC.EG.EGSTRESC.ECG22 条件 / 2 值
──────────────────────────────────────────────────────────
  💡 key=WhereClauseOID；多条件即 AND；IN 多值折叠为 values；QN

## Comments

In [6]:
# ============================================================================
#  提取部门级 comments → 与源文件同名的 .yaml(全量, 不过滤)
#  key = CommentOID 去掉 'COM.' 前缀(对齐 domain yaml 的 comment 裸名引用)
#  description 双语; doc/page 引用有才写。项目筛选留给下游按 datasets/*.yaml 取子集。
#  依赖: openpyxl pyyaml
# ============================================================================
import yaml, openpyxl, os, time
from pathlib import Path
from collections import OrderedDict

# ---------- 配置 ----------
SRC        = COMMENTS_XLSX        # config cell: DEPT_DIR / dept_files.comments_xlsx
SHEET      = "Comments"
OID_PREFIX = "COM."
OUT        = COMMENTS_LIB_YAML    # config cell: DEPT_DIR / (stem + ".yaml")

# 源列 -> 输出字段(顺序即 yaml 内字段顺序; 仅在有值时写入)
FIELDS = [
    ("Description",  "description_en"),  ("DescriptionCN","description_cn"),
    ("DocsleafID",   "doc_id"),          ("PageRefType",  "page_ref_type"),
    ("PageRefs",     "page_refs"),       # 文档/页码引用：部门版常空, 项目版才填
]
_t0 = time.time()

wb=openpyxl.load_workbook(SRC, read_only=True, data_only=True)
ws=wb[SHEET]; it=ws.iter_rows(values_only=True); hdr=next(it)
H, g = make_getter(hdr)

out=OrderedDict(); dups=[]; skipped=0
for r in it:
    oid=g(r,"CommentOID")
    if not oid:                                              # 空行跳过
        skipped+=1; continue
    key=str(oid)[len(OID_PREFIX):] if str(oid).startswith(OID_PREFIX) else str(oid)
    if key in out:                                           # 重复 OID: 保留首次
        dups.append(key); continue
    e=OrderedDict()
    for col, ok in FIELDS:                                   # 单次取值, 数据驱动
        v=g(r,col)
        if v: e[ok]=v
    out[key]=e

out=OrderedDict(sorted(out.items()))
with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# 部门级 comments 全量提取自 {SRC}(key = CommentOID 去 '{OID_PREFIX}', description 双语)。项目层面按 datasets/*.yaml 的 comment 引用筛选。\n")
    yaml.dump(out, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=200)

# ---------- 统计 ----------
n       = len(out)
has_en  = sum(1 for e in out.values() if e.get("description_en"))
has_cn  = sum(1 for e in out.values() if e.get("description_cn"))
bil     = sum(1 for e in out.values() if e.get("description_en") and e.get("description_cn"))
has_doc = sum(1 for e in out.values() if e.get("doc_id") or e.get("page_ref_type") or e.get("page_refs"))
no_desc = sum(1 for e in out.values() if not (e.get("description_en") or e.get("description_cn")))

# ---------- 打印确认（_pad 来自 config cell；_BAR/_SUB/_sz 本地） ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  💬  部门级 Comments 提取")
print(_BAR)
print(f"  📥 {_pad('输入',6)}{SRC}   ({_sz(SRC)})")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s   ·   跳过空行 {skipped}")
print(_SUB)
print(f"  📊 comment {n} 条")
print(f"  🌐 内容覆盖")
print(f"       · 有英文描述 description    {has_en:>4} / {n}")
print(f"       · 有中文描述 descriptionCN  {has_cn:>4} / {n}")
print(f"       · 双语 (中英都有)           {bil:>4} / {n}")
print(f"       · 含文档/页码引用           {has_doc:>4} / {n}   {'(部门版通常为空)' if has_doc==0 else ''}")
if no_desc: print(f"       · ⚠ 无任何描述              {no_desc:>4}")
print(_SUB)
print("  💡 key = CommentOID 去 'COM.' 前缀；项目层按 datasets/*.yaml 的 comment 引用筛子集")
print(_BAR)
if dups: print(f"  ⚠ [重复OID] 保留首次: {sorted(set(dups))}")


══════════════════════════════════════════════════════════
  💬  部门级 Comments 提取
══════════════════════════════════════════════════════════
  📥 输入  dept_std\comments-studyname_sdtm_v1.1.xlsx   (22.4 KB)
  📤 输出  dept_std\comments-studyname_sdtm_v1.1.yaml   (4.4 KB)
  ⏱  耗时  0.0 s   ·   跳过空行 0
──────────────────────────────────────────────────────────
  📊 comment 19 条
  🌐 内容覆盖
       · 有英文描述 description      19 / 19
       · 有中文描述 descriptionCN    19 / 19
       · 双语 (中英都有)             19 / 19
       · 含文档/页码引用              2 / 19   
──────────────────────────────────────────────────────────
  💡 key = CommentOID 去 'COM.' 前缀；项目层按 datasets/*.yaml 的 comment 引用筛子集
══════════════════════════════════════════════════════════


# 提取项目文件

## aCRF

In [7]:
# ============================================================================
#  extract_acrf —— 从注释版 aCRF(acrf.pdf) 提取 变量→页码 映射
#  仅产出 acrf_pages.yaml：{域: {变量: [页码]}}，供生成 yaml 时填 origin.crf_pages
#  注释结构：域横幅 "AE (不良事件)" + 主变量 "AETERM" + SUPP限定符 "EXTRT in SUPPAE"
#  依赖：pymupdf pyyaml
# ============================================================================
import fitz, re, yaml, os, time
from pathlib import Path
from collections import defaultdict, OrderedDict

# ---------- 配置 ----------
ACRF_PATH = ACRF_PDF          # config cell: SDTM_DIR / study_files.acrf_pdf
OUT_YAML  = ACRF_PAGES_YAML   # config cell: STUDY_DIR / (stem + ".yaml")

DOMAIN_RE = re.compile(r'^([A-Z]{2}[A-Z0-9]{0,6})\s*[\(（]')                         # "AE (不良事件)"
SUPP_RE   = re.compile(r'^([A-Z][A-Z0-9]{1,7})\s+in\s+(SUPP[A-Z0-9]{1,6})$', re.I)   # "EXTRT in SUPPAE"
VAR_RE    = re.compile(r'^[A-Z][A-Z0-9]{1,7}$')                                      # 裸 SDTM 变量

def domain_for(y, banners):                           # 裸变量按 y 找上方最近域横幅(多域页)
    dom = None
    for by, bn in banners:
        if by <= y + 2: dom = bn
        else: break
    return dom

_t0 = time.time()
doc = fitz.open(ACRF_PATH)
pages_map = defaultdict(lambda: defaultdict(set))     # domain -> var -> {pages}

# 注释解析计数(互斥分类, 仅统计用, 不影响 pages_map)
n_pages_ann=n_ann=n_banner=n_supp=n_var_match=n_var_nodomain=n_other=0
for pi in range(doc.page_count):
    pg = doc[pi]; page_no = pi + 1
    anns = [(a.rect.y0, (a.info.get('content') or '').strip()) for a in (pg.annots() or [])]
    anns = [(y, c) for (y, c) in anns if c]
    if not anns: continue
    n_pages_ann += 1

    # 本页域横幅(按 y)，给裸变量定位归属域(多域页取上方最近横幅)
    banners = sorted([(y, DOMAIN_RE.match(c).group(1)) for (y, c) in anns if DOMAIN_RE.match(c)])

    for (y, c) in anns:
        n_ann += 1
        if DOMAIN_RE.match(c):                                    # 域横幅(仅用于定位, 不入表)
            n_banner += 1; continue
        m = SUPP_RE.match(c)
        if m:                                                    # SUPP 限定符(自带域名)
            pages_map[m.group(2).upper()][m.group(1).upper()].add(page_no); n_supp += 1
        elif VAR_RE.match(c):                                    # 裸变量(banner/派生/[NOT]天然排除)
            dom = domain_for(y, banners)
            if dom: pages_map[dom][c].add(page_no); n_var_match += 1
            else:   n_var_nodomain += 1                          # 匹配变量名但本页无域横幅
        else:
            n_other += 1                                         # 派生/未提交/自由文本等

# ---------- 写出 ----------
out = OrderedDict((dom, OrderedDict((v, sorted(pages_map[dom][v])) for v in sorted(pages_map[dom])))
                  for dom in sorted(pages_map))
with open(OUT_YAML, 'w', encoding='utf-8') as f:
    f.write("# 从 aCRF 注释提取的 变量→页码 映射(权威来源)。供生成 yaml 时填 origin.crf_pages。\n")
    yaml.dump(out, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=100)

# ---------- 统计 ----------
n_dom      = len(out)
n_supp_dom = sum(1 for d in out if d.startswith("SUPP"))
n_main     = n_dom - n_supp_dom
n_var      = sum(len(v) for v in out.values())                         # 唯一 变量条目
n_pref     = sum(len(p) for d in out.values() for p in d.values())     # 页引用总数
n_multi    = sum(1 for d in out.values() for p in d.values() if len(p)>1)
_top       = sorted(((d,len(v)) for d,v in out.items() if not d.startswith("SUPP")),
                    key=lambda kv: kv[1], reverse=True)[:5]

# ---------- 打印确认（_pad 来自 config cell；_BAR/_SUB/_sz/_samp 本地） ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
def _samp(dom):                                                 # 抽样: 变量[页,页] …
    d=out.get(dom)
    if not d: return f"{dom} → (无)"
    items=" ".join(f"{v}[{','.join(map(str,p))}]" for v,p in list(d.items())[:5])
    more = "" if len(d)<=5 else f"  …(+{len(d)-5})"
    return f"{dom} → {items}{more}"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  📝  aCRF 注释提取 (变量→页码)")
print(_BAR)
print(f"  📥 {_pad('输入',6)}{ACRF_PATH}   ({_sz(ACRF_PATH)})")
print(f"  📤 {_pad('输出',6)}{OUT_YAML}   ({_sz(OUT_YAML)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s")
print(f"  📄 {_pad('PDF',6)}{doc.page_count} 页 (含注释 {n_pages_ann} 页)")
print(_SUB)
print(f"  📊 域 {n_dom} 个 (主域 {n_main} · SUPP {n_supp_dom})   ·   变量条目 {n_var} 个   ·   页引用 {n_pref} 处")
print(f"  🔖 注释解析 (共 {n_ann} 条非空注释)")
print(f"       · 域横幅 banner        {n_banner:>4}")
print(f"       · SUPP 限定符          {n_supp:>4}")
print(f"       · 裸变量 (已归域)       {n_var_match:>4}")
if n_var_nodomain: print(f"       · ⚠ 裸变量 (无归属域)  {n_var_nodomain:>4}")
print(f"       · 其它 (派生/未提交等)  {n_other:>4}")
if n_multi: print(f"  📑 跨多页变量          {n_multi}")
print(f"  🏆 变量最多的主域 (top 5)")
for d,c in _top: print(f"       · {_pad(d,8)}{c:>4}")
print(_SUB)
print("  🔍 抽样校验")
print(f"       {_samp('AE')}")
print(f"       {_samp('SUPPAE')}")
print(_BAR)


══════════════════════════════════════════════════════════
  📝  aCRF 注释提取 (变量→页码)
══════════════════════════════════════════════════════════
  📥 输入  m5\datasets\tabulations\sdtm\acrf.pdf   (6.6 MB)
  📤 输出  study_std\acrf.yaml   (7.4 KB)
  ⏱  耗时  0.3 s
  📄 PDF   207 页 (含注释 83 页)
──────────────────────────────────────────────────────────
  📊 域 40 个 (主域 22 · SUPP 18)   ·   变量条目 232 个   ·   页引用 483 处
  🔖 注释解析 (共 1214 条非空注释)
       · 域横幅 banner         100
       · SUPP 限定符           174
       · 裸变量 (已归域)        349
       · ⚠ 裸变量 (无归属域)     2
       · 其它 (派生/未提交等)   589
  📑 跨多页变量          77
  🏆 变量最多的主域 (top 5)
       · AE        15
       · CM        13
       · TU        12
       · PR        10
       · EX         8
──────────────────────────────────────────────────────────
  🔍 抽样校验
       AE → AEACN[185] AECONTRT[185] AEENDTC[185] AEOUT[185] AEREL[185]  …(+10)
       SUPPAE → AEACNC[185] AEACNE[185] AEACNG[185] AEACNP[185] AEACNPA[185]  …(+8)
═════════════════════════════════════════

## Codelist File

In [8]:
# ============================================================================
#  提取 standard codelist → standard_codelist.yaml (中/英文提交值 ↔ NCI 桥)
#  本版读「项目层面」codelist 文件: 自动识别项目标记列(非标准列且值几乎全为 X),
#  只取该列=X 的行; 识别不到则不过滤(回到全量, 等价旧部门版)。
#  (码表级字段 CTLabel/CTLabelCN/ParentAlias/CTExtensible/CTDictionary 每行都有, 故先过滤再分组安全)
#  依赖: openpyxl pyyaml
# ============================================================================
import re, yaml, openpyxl, os, time
from pathlib import Path
from collections import OrderedDict, defaultdict, Counter

# ---------- 配置 ----------
STDCL_SRC = STDCL_XLSX    # config cell: study_files.stdcl_xlsx
SHEET     = "codelist"
OUT       = STDCL_YAML    # config cell: STUDY_DIR / "standard_codelist.yaml"
# P21 标准码表列(用于排除, 识别项目标记列时这些列不算)
KNOWN_COLS = {"ADAMCTFL","CTName","CTLabel","CTLabelCN","Type","CharCode","CharCodeCN",
  "LowNumCode","HighNumCode","Decode","DecodeCN","EnumeratedItem","CTRank","CTOrderNumber",
  "ParentAlias","CTExtensible","CTAlias","StandardOID","IsNonStandard","CTDictionary",
  "CTDictVersion","LocalRef","HyperlnRef"}
ITEM_STR = [("Decode","decode_en"), ("DecodeCN","decode_cn"), ("CTAlias","nci_code")]  # item 级简单字段
_t0 = time.time()

wb=openpyxl.load_workbook(STDCL_SRC, read_only=True, data_only=True)
ws=wb[SHEET]; it=ws.iter_rows(values_only=True); hdr=next(it)
H, g = make_getter(hdr)
data=list(it)                                            # 物化(需两遍: 先识别标记列, 再分组)

# ---------- 自动识别项目标记列: 非标准列 + 非空值几乎全为 X ----------
def detect_project_flag(hdr, rows):
    best=None
    for ci,h in enumerate(hdr):
        if h is None or h in KNOWN_COLS: continue
        vals=[r[ci] for r in rows if ci<len(r) and r[ci] not in (None,"")]
        if not vals: continue
        x_ratio=sum(str(v).strip().upper()=="X" for v in vals)/len(vals)
        if x_ratio < 0.8: continue                       # 必须基本全是 X 才算标记列
        idlike=bool(re.search(r"[A-Za-z]", str(h)) and re.search(r"\d", str(h)))  # 研究号样式优先
        score=(round(x_ratio,3), idlike)
        if best is None or score>best[0]: best=(score, ci, h)
    return (best[2], best[1]) if best else (None, None)

PROJECT_FLAG, flag_ci = detect_project_flag(hdr, data)

def in_project(r):
    if flag_ci is None: return True
    v=r[flag_ci] if flag_ci<len(r) else None
    return str(v or "").strip().upper()=="X"

# ---------- 先按 项目标记=X 过滤, 再按 CTName 分组 ----------
groups=defaultdict(list); n_kept=0
for r in data:
    if not in_project(r): continue
    nm=g(r,"CTName")
    if nm: groups[nm].append(r); n_kept+=1

out=OrderedDict(); n_item=0
for nm,rows in groups.items():
    r0=rows[0]                                            # 码表级字段每行都有, 取任一 X 行即可
    c=OrderedDict()
    lbl=g(r0,"CTLabel")
    if lbl: c["name_en"]=lbl
    lblcn=g(r0,"CTLabelCN")
    if lblcn: c["name_cn"]=lblcn
    dic=g(r0,"CTDictionary")
    if dic:                                               # 外部字典: 无枚举项
        c["type"]="external"; c["dictionary"]=dic
        ver=g(r0,"CTDictVersion")
        if ver: c["version"]=str(ver)
        href=g(r0,"HyperlnRef")
        if href: c["href"]=href
        out[nm]=c; continue
    pa=g(r0,"ParentAlias")
    if pa: c["nci_code"]=pa                               # 码表级 NCI
    ext=g(r0,"CTExtensible")
    if ext is not None: c["extensible"]=(str(ext).strip().lower()=="yes")
    items=[]
    for rr in rows:                                       # 仅 X 行
        vcn=g(rr,"CharCodeCN"); ven=g(rr,"CharCode")
        if not (vcn or ven): continue
        e=OrderedDict()
        if vcn: e["value_cn"]=vcn
        if ven: e["value_en"]=ven
        for col, ok in ITEM_STR:                          # decode_en / decode_cn / nci_code, 单次取值
            val=g(rr,col)
            if val: e[ok]=val
        o=g(rr,"CTOrderNumber") or g(rr,"CTRank")
        if o is not None:
            try: e["order"]=int(o)
            except: pass
        items.append(e); n_item+=1
    if items: c["items"]=items
    out[nm]=c

note = f"仅项目标记列 {PROJECT_FLAG}=X 的行" if PROJECT_FLAG else "全量(未识别项目标记列)"
with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# 项目层面标准码表库提取({note})。中/英文提交值↔NCI 桥 + 码表中文名 + 码表级NCI + 可扩展性。\n")
    yaml.dump(out, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=200)

# ---------- 统计 ----------
enum     = [(k,v) for k,v in out.items() if v.get("type")!="external"]
ext      = [(k,v) for k,v in out.items() if v.get("type")=="external"]
n_cl, n_enum, n_ext = len(out), len(enum), len(ext)
allitems = [e for _,v in enum for e in v.get("items",[])]
ext_dist = Counter((v["dictionary"], v.get("version","")) for _,v in ext)
x_true   = sum(1 for _,v in enum if v.get("extensible") is True)
x_false  = sum(1 for _,v in enum if v.get("extensible") is False)
x_absent = n_enum - x_true - x_false
has_cn   = sum(1 for v in out.values() if v.get("name_cn"))
cl_nci   = sum(1 for _,v in enum if v.get("nci_code"))
it_vcn   = sum(1 for e in allitems if e.get("value_cn"))
it_ven   = sum(1 for e in allitems if e.get("value_en"))
it_bil   = sum(1 for e in allitems if e.get("value_cn") and e.get("value_en"))
it_nci   = sum(1 for e in allitems if e.get("nci_code"))
_top     = sorted(((k,len(v.get("items",[]))) for k,v in enum), key=lambda kv: kv[1], reverse=True)[:5]

# ---------- 打印确认（_pad 来自 config cell；_BAR/_SUB/_sz 本地） ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  🗂️  项目标准码表库提取")
print(_BAR)
print(f"  📥 {_pad('输入',8)}{STDCL_SRC}   ({_sz(STDCL_SRC)})")
print(f"  📤 {_pad('输出',8)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',8)}{time.time()-_t0:.1f} s")
if PROJECT_FLAG:
    nX=sum(1 for r in data if in_project(r))
    print(f"  🏷 {_pad('标记列',8)}{PROJECT_FLAG!r}  (标 X 共 {nX} 行)")
    print(f"  🔻 {_pad('过滤',8)}全表 {len(data)} 行 → 保留 {n_kept} 行")
else:
    print(f"  🏷 {_pad('标记列',8)}未识别 → 不过滤, 提取全量(等价部门版)")
print(_SUB)
print(f"  📊 码表 {n_cl} 个 (枚举 {n_enum} · 外部 {n_ext})   ·   项 {n_item} 个")
if ext_dist:
    print(f"  📚 外部字典")
    for (d,v),cnt in ext_dist.most_common():
        print(f"       · {_pad(str(d),10)}{v or '(无版本)'}" + (f"   ×{cnt}" if cnt>1 else ""))
print(f"  🔧 枚举码表扩展性")
print(f"       · 可扩展    {x_true:>4}")
print(f"       · 不可扩展  {x_false:>4}")
if x_absent: print(f"       · 未标注    {x_absent:>4}")
print(f"  🌐 编码覆盖")
print(f"       · 码表有中文名 name_cn      {has_cn:>4} / {n_cl}")
print(f"       · 码表有 NCI (ParentAlias)  {cl_nci:>4} / {n_enum}")
print(f"       · 项有 value_cn             {it_vcn:>4} / {n_item}")
print(f"       · 项有 value_en             {it_ven:>4} / {n_item}")
print(f"       · 项双语 (中英都有)          {it_bil:>4} / {n_item}")
print(f"       · 项有 NCI (CTAlias)        {it_nci:>4} / {n_item}")
print(f"  🏆 最大枚举码表 (按项数)")
for k,cnt in _top: print(f"       · {_pad(k,12)}{cnt:>4}")
print(_BAR)


══════════════════════════════════════════════════════════
  🗂️  项目标准码表库提取
══════════════════════════════════════════════════════════
  📥 输入    study_std\standard-codelist-SHR-1234-III-301_for_define.xlsx   (507.9 KB)
  📤 输出    study_std\standard_codelist.yaml   (190.4 KB)
  ⏱  耗时    0.9 s
  🏷 标记列  'SHR_1234_III_301'  (标 X 共 1232 行)
  🔻 过滤    全表 5282 行 → 保留 1232 行
──────────────────────────────────────────────────────────
  📊 码表 174 个 (枚举 169 · 外部 5)   ·   项 1226 个
  📚 外部字典
       · ISO 3166-1 Alpha-32013-11-15
       · WHODRUG   GLOBALB3Mar23
       · MEDDRA    28.0
       · CTCAE     5.0
       · ISO 21090 NullFlavor enumeration2011-02
  🔧 枚举码表扩展性
       · 可扩展      71
       · 不可扩展    17
       · 未标注      81
  🌐 编码覆盖
       · 码表有中文名 name_cn       174 / 174
       · 码表有 NCI (ParentAlias)    92 / 169
       · 项有 value_cn             1223 / 1226
       · 项有 value_en             1200 / 1226
       · 项双语 (中英都有)          1197 / 1226
       · 项有 NCI (CTAlias)         521 / 1226
  🏆 最大枚举码表 

## SDTM Spec / Dataset

In [9]:
# ============================================================================
#  build_domain_yaml (全域版) —— 遍历 SDTM_DIR 下所有 xpt, 每个主域配对其 SUPP,
#  各产出 datasets/{域}.yaml。首次生成即注入值级(value_list):
#    SUPP QVAL  ← supp{域}.xpt 的 qnam 现取(数据驱动, 原有逻辑)
#    主域非SUPP ← 部门 valuelevel/whereclause 库, 按本域 xpt 实际数据筛选(新增)
#  注释: comment=注释 key(→COM.xxx) · comment_text=研究专属正文(spec Study Derivation, 有才写)
#  码表名: Controlled Terms 原值(仅 MedDRA→MEDDRACT 别名), 不加 域. 前缀
#  依赖：pyyaml openpyxl pyreadstat pandas
# ============================================================================
import re, os, glob, time, yaml, openpyxl, pandas as pd
from pathlib import Path
from collections import OrderedDict, defaultdict
_t0 = time.time()

# ---------- 配置（均来自 config cell） ----------
SPEC_XLSX  = SPEC_XLSX          # study_files.spec_xlsx
STUDY_ID   = STUDY_ID         # config cell: 从 SPEC_XLSX 文件名推导
ACRF_PAGES = ACRF_PAGES_YAML    # STUDY_DIR / (acrf_pdf stem + ".yaml")
OUT_DIR    = DATASETS_DIR       # output_files.datasets_dir
VLM_LIB   = VALUELEVEL_YAML    # DEPT_DIR / (valuelevel_xlsx stem + ".yaml")
WC_LIB    = WHERECLAUSE_YAML   # DEPT_DIR / ("whereclause-" + ...)
# SDTM_DIR / LANG / SUF 直接沿用 config cell 已暴露的同名变量

CL_ALIAS  = {"MedDRA":"MEDDRACT", "WHODrug":"DRUGDICT"}
# ---------- 语言相关映射/列名（LANG/SUF 来自 config cell；EN 模式不得出现中文） ----------
CLASS_MAP = ({"EVENTS":"事件类","INTERVENTIONS":"干预类","FINDINGS":"发现类","FINDINGS ABOUT":"发现关于",
              "SPECIAL-PURPOSE":"SPECIAL-PURPOSE","TRIAL DESIGN":"试验设计","RELATIONSHIP":"关联数据",
              "STUDY REFERENCE":"研究参考"} if LANG=="CN" else {})        # EN: class 原样英文
PURPOSE_MAP     = {"Tabulation":"列表"} if LANG=="CN" else {}             # EN: 原样 "Tabulation"
DEFAULT_PURPOSE = "列表" if LANG=="CN" else "Tabulation"
DESC_COL   = "DescriptionCN"    if LANG=="CN" else "Description"
STRUCT_COL = "StructureCN"      if LANG=="CN" else "Structure"
VLABEL_COL = "Variable LabelCN" if LANG=="CN" else "Variable Label"
SUPP_DESC_FALLBACK = "[domain name]补充修饰语数据集" if LANG=="CN" else "[domain name] Supplemental Qualifiers"
QORIG_MAP = {"CRF":"Collected","DERIVED":"Derived","ASSIGNED":"Assigned","PROTOCOL":"Protocol",
             "PREDECESSOR":"Predecessor","EDT":"Collected","EDC":"Collected"}
SUPP_TEMPLATE = [
  ("STUDYID",  dict(origin=("Protocol","Sponsor"),  mandatory=True,  key=1)),
  ("RDOMAIN",  dict(origin=("Assigned","Sponsor"),  mandatory=True,  key=2, codelist="DOMAIN")),
  ("USUBJID",  dict(origin=("Derived","Sponsor"),   mandatory=True,  key=3, method="USUBJID")),
  ("IDVAR",    dict(origin=("Assigned","Sponsor"),  mandatory=False, key=4)),
  ("IDVARVAL", dict(origin=("Assigned","Sponsor"),  mandatory=False, key=5)),
  ("QNAM",     dict(origin=("Assigned","Sponsor"),  mandatory=True,  key=6)),
  ("QLABEL",   dict(origin=("Assigned","Sponsor"),  mandatory=True)),
  ("QVAL",     dict(origin=("Assigned","Sponsor"),  mandatory=True,  qval=True)),
  ("QORIG",    dict(origin=("Assigned","Sponsor"),  mandatory=True)),
  ("QEVAL",    dict(origin=("Assigned","Sponsor"),  mandatory=False)),
]
# SUPP-- 10 个标准变量的标签：CDISC SDTMIG 固定（与上面 SUPP_TEMPLATE 一一对应），仅在 xpt 无自带 label 时兜底
SUPP_LABELS = ({"STUDYID":"研究标识符","RDOMAIN":"关联域名缩写","USUBJID":"受试者唯一标识符",
  "IDVAR":"标识变量","IDVARVAL":"标识变量值","QNAM":"修饰语变量名称","QLABEL":"修饰语变量标签",
  "QVAL":"数据值","QORIG":"来源","QEVAL":"评估人员"} if LANG=="CN" else
  {"STUDYID":"Study Identifier","RDOMAIN":"Related Domain Abbreviation","USUBJID":"Unique Subject Identifier",
  "IDVAR":"Identifying Variable","IDVARVAL":"Identifying Variable Value","QNAM":"Qualifier Variable Name",
  "QLABEL":"Qualifier Variable Label","QVAL":"Data Value","QORIG":"Origin","QEVAL":"Evaluator"})
# SUPP-- 结构描述：CDISC 固定文案，每个研究的每个 SUPP 域通用
SUPP_STRUCTURE = "一个域一个修饰变量一条记录" if LANG=="CN" else "One record per IDVAR, IDVARVAL, and QNAM value per subject"
# 部门 spec 模板的标准列名集合：detect_flag_col 用它排除标准列，避免把恰好多为 X 的标准列误判成项目标记列
# （反映部门模板 schema，非 CDISC 标准；模板增改列时需同步维护）
KNOWN_COLS = {"Variable Name","Variable Label","Variable LabelCN","Key","Type","Data Type","Length",
  "Format","Controlled Terms","Origin","CRF Page","Source","Derivation/Comment","Flag","Flag_SuppIDVar",
  "Role","CDISC Notes","HR Notes","Core","HR Core","Method OID","ValueList OID","Comment OID","Study Derivation",
  "Dataset","Description","DescriptionCN","Class","Structure","StructureCN","Purpose","Keys","Repeating",
  "IsReferenceData","Location","StandardOID","IsNonStandard","HasNoData","Order"}

# ---------- 工具(与域无关, 全局一份；representer/cidx/cell/Flow 已在 config cell 定义) ----------
def rows_of(ws): return list(ws.iter_rows(values_only=True))
def num_length(series):
    best=0
    for v in series.dropna():
        try:
            f=float(v)
            d=len(str(abs(int(f)))) if f.is_integer() else len(re.sub(r'\D','',repr(f)))
        except Exception:
            d=len(str(v))
        best=max(best,d)
    return best or None
def detect_flag_col(rows, study_id):
    hdr=rows[0]; norm=lambda s: re.sub(r"[^A-Za-z0-9]","",str(s)).upper(); sid=norm(study_id); best=None
    for ci,h in enumerate(hdr):
        if h is None: continue
        vals=[r[ci] for r in rows[1:] if ci<len(r) and r[ci] not in (None,"")]
        if not vals: continue
        x_ratio=sum(str(v).strip() in ("X","x") for v in vals)/len(vals)
        score=(2 if norm(h)==sid else 0)+(1 if x_ratio>=0.8 and h not in KNOWN_COLS else 0)
        if score>0 and (best is None or score>best[1]): best=(ci,h,score,round(x_ratio,2))
    return best
def crf_of(dom, name):
    return [str(p) for p in acrf_pages.get(dom, {}).get(name, [])]
def valid_ct(ct):
    return bool(ct) and bool(re.match(r"^[A-Za-z][A-Za-z0-9_]*$", str(ct).strip()))
def resolve_cl(ct):
    if not ct: return None
    return CL_ALIAS.get(str(ct), str(ct))

# ---------- 值级工具(全局一份) ----------
def dnorm(s):                                             # 数据/比较值归一(去空白, 解码 bytes)
    if s is None: return ""
    if isinstance(s,bytes): s=s.decode("utf-8","ignore")
    if isinstance(s,float) and pd.isna(s): return ""
    return str(s).strip()
def apply_where(df, conds, warn):                         # 条件 AND → 掩码; None=无法判定
    mask=pd.Series(True, index=df.index)
    for c in conds:
        var=c["variable"]
        if var not in df.columns: warn.append(f"判别变量 {var} 不在列"); return None
        col=df[var].map(dnorm); vals=[dnorm(v) for v in c["values"]]; comp=c["comparator"]
        if   comp=="EQ":    m=col==vals[0]
        elif comp=="IN":    m=col.isin(vals)
        elif comp=="NE":    m=col!=vals[0]
        elif comp=="NOTIN": m=~col.isin(vals)
        else: warn.append(f"未知 comparator {comp}"); return None
        mask&=m
    return mask
def make_vl_entry(item, var_label, df, warn):             # 库里一条值项 → value_list 条目(按数据筛)
    w=item.get("where")
    if w not in WC_DATA: warn.append(f"值级 where 缺失:{w}"); return None
    conds=WC_DATA[w]
    mask=apply_where(df, conds, warn)
    if mask is None: return None
    if int(mask.sum())==0: warn.append(f"值级丢弃 {w}(数据命中0条)"); return None
    e=OrderedDict()
    e["where"]=[OrderedDict([("variable",c["variable"]),("comparator",c["comparator"]),
                             ("values",list(c["values"]))]) for c in conds]   # 内联(去 dataset/comment)
    if item.get("method"):   e["method"]=re.sub(r"^MT\.","",str(item["method"]))   # 去 MT. 对齐 methods.yaml
    if item.get("codelist"): e["codelist"]=item["codelist"]
    if var_label:            e["label"]=var_label                              # 用变量自身 label(同反推)
    if item.get("origin"):
        o=item["origin"]; oo=OrderedDict(type=o.get("type"))
        if o.get("source"): oo["source"]=o["source"]
        e["origin"]=oo
    e["mandatory"]=bool(item.get("mandatory", False))
    de=item.get("description"+SUF) or item.get("description_en") or item.get("description_cn")
    if de: e["description"]=de
    return e

# ---------- 读一次: aCRF / 值级库 / 工作簿+TOC ----------
# LANG / SUF 已由 config cell 暴露，无需重读 config.yaml
acrf_pages={}; _n_acrf=0
if Path(ACRF_PAGES).exists():
    acrf_pages=yaml.safe_load(Path(ACRF_PAGES).read_text(encoding="utf-8")) or {}
    _n_acrf=sum(len(v) for v in acrf_pages.values())
VLM_DATA=(yaml.safe_load(open(VLM_LIB,encoding="utf-8")) or {}) if Path(VLM_LIB).exists() else {}
WC_DATA =(yaml.safe_load(open(WC_LIB ,encoding="utf-8")) or {}) if Path(WC_LIB).exists() else {}
_vlm_found = bool(VLM_DATA)
wb=openpyxl.load_workbook(SPEC_XLSX, read_only=True, data_only=True)
toc=rows_of(wb["TOC"]); ti=cidx(toc[0])

# ============================================================================
#  单域构建函数
# ============================================================================
def build_domain(DOMAIN, XPT_PATH, SUPP_XPT):
    warn=[]
    OUT = f"{DOMAIN.lower()}.yaml"

    # ---- 读主 xpt(结构层) ----
    xpt_meta={}; xpt_df=None
    if Path(XPT_PATH).exists():
        import pyreadstat
        xpt_df, m = pyreadstat.read_xport(XPT_PATH)
        sw=getattr(m,"variable_storage_width",{}) or {}
        for i,n in enumerate(m.column_names):
            xpt_meta[n]=dict(order=i+1, label=m.column_labels[i] if m.column_labels else None, width=sw.get(n))
    else:
        warn.append(f"未找到 {XPT_PATH} → 结构回退 xlsx(Length 可能占位)")

    def length_for(name, vtype, xlsx_len):
        t=str(vtype or "")
        if t in ("date","datetime","time"): return None
        if t in ("integer","float","number"):
            if xpt_df is not None and name in xpt_df.columns:
                nl=num_length(xpt_df[name])
                if nl: return str(nl)
            return str(xlsx_len) if xlsx_len is not None else None
        w=xpt_meta.get(name,{}).get("width")
        if w is not None: return str(w)
        if not xpt_meta and xlsx_len is not None: warn.append(f"{name}: Length 回退 xlsx({xlsx_len})")
        return str(xlsx_len) if xlsx_len is not None else None

    # ---- 数据集级 ----
    if DOMAIN not in wb.sheetnames:
        warn.append(f"spec 无 {DOMAIN} sheet → 跳过该域"); return None, 0, 0, 0, False, warn, {}
    dr=next((r for r in toc[1:] if cell(r,ti,"Dataset")==DOMAIN), None)
    if dr is None:
        warn.append(f"TOC 无 {DOMAIN} 行 → 跳过该域"); return None, 0, 0, 0, False, warn, {}
    ds=OrderedDict(domain=DOMAIN)
    ds["label"]=cell(dr,ti,DESC_COL)
    cls=cell(dr,ti,"Class"); ds["class"]=CLASS_MAP.get(cls,cls)
    pur=cell(dr,ti,"Purpose"); ds["purpose"]=PURPOSE_MAP.get(pur,pur)
    ds["repeating"]=(str(cell(dr,ti,"Repeating")).upper()=="Y")
    ds["is_reference_data"]=(str(cell(dr,ti,"IsReferenceData")).upper()=="Y")
    ds["structure"]=cell(dr,ti,STRUCT_COL)
    ds["sas_dataset"]=DOMAIN
    ds["keys"]=[k.strip() for k in str(cell(dr,ti,"Keys")).split(",")] if cell(dr,ti,"Keys") else []
    ds["archive"]=cell(dr,ti,"Location")
    # has_no_data: xpt 存在且行数=0 → True；xpt 不存在不置(无法判断)
    if xpt_df is not None and len(xpt_df) == 0:
        ds["has_no_data"] = True
        warn.append(f"{DOMAIN}: xpt 行数=0 → has_no_data=True")

    # ---- 变量级 ----
    sh=rows_of(wb[DOMAIN]); si=cidx(sh[0])
    flag=detect_flag_col(sh, STUDY_ID)
    if not flag:
        warn.append(f"{DOMAIN}: 未识别项目标记列 → 跳过该域"); return None, 0, 0, 0, False, warn, {}
    fci,fname,fscore,fratio=flag

    SUPP_VL=f"IT.SUPP{DOMAIN}.QVAL"
    variables=OrderedDict(); supp_xlsx=OrderedDict()
    for ridx,r in enumerate(sh[1:],1):
        name=cell(r,si,"Variable Name")
        if not name: continue
        is_supp=str(cell(r,si,"ValueList OID") or "").startswith(SUPP_VL)
        flag_val=str(cell(r,si,fname) or "").strip()
        _ct=cell(r,si,"Controlled Terms")
        ct=resolve_cl(_ct) if valid_ct(_ct) else None
        mo=cell(r,si,"Method OID");  method=re.sub(r"^MT\.","",str(mo)) if mo else None
        co=cell(r,si,"Comment OID"); comment=re.sub(r"^COM\.","",str(co)) if co else None
        sd=cell(r,si,"Study Derivation"); comment_text=str(sd).strip() if (co and sd) else None

        if is_supp:
            if not flag_val: continue
            supp_xlsx[name]=dict(codelist=ct, method=method, comment=comment, comment_text=comment_text,
                                 source=cell(r,si,"Source"), origin_x=cell(r,si,"Origin"), label_x=cell(r,si,"Variable LabelCN"))
            continue
        if flag_val not in ("X","x"): continue

        v=OrderedDict()
        v["label"]=cell(r,si,VLABEL_COL) or xpt_meta.get(name,{}).get("label")
        v["type"]=cell(r,si,"Data Type")
        L=length_for(name, v["type"], cell(r,si,"Length"))
        if L is not None: v["length"]=L
        v["order"]=xpt_meta.get(name,{}).get("order", ridx)
        k=cell(r,si,"Key")
        if k is not None and str(k).isdigit(): v["key_sequence"]=int(k)
        v["mandatory"]=(str(cell(r,si,"Core")).strip()=="Req")
        origin=OrderedDict(type=cell(r,si,"Origin"))
        src=cell(r,si,"Source")
        if src: origin["source"]=src
        pages=crf_of(DOMAIN, name)
        if pages: origin["crf_pages"]=pages
        elif str(origin["type"]).strip()=="Collected": warn.append(f"{name}: Collected 但 aCRF({DOMAIN}) 无页码")
        v["origin"]=Flow(origin) if not pages else origin
        if ct: v["codelist"]=ct
        if method: v["method"]=method
        if comment: v["comment"]=comment
        if comment_text: v["comment_text"]=comment_text
        variables[name]={kk:vv for kk,vv in v.items() if vv not in (None,[],")")}

    ds["variables"]=variables

    # ---- 值级注入 ----
    vl_report={"injected":[], "dropped":[], "uncovered":[]}
    if VLM_DATA:
        dom_vlmoids=[kk for kk in VLM_DATA if kk.split(".")[1]==DOMAIN]
        if dom_vlmoids and xpt_df is None:
            vl_report["dropped"].append((DOMAIN, f"无 {XPT_PATH}, 跳过本域值级注入"))
        elif dom_vlmoids:
            claimed=defaultdict(set)
            for vlmoid in dom_vlmoids:
                var=vlmoid.split(".",2)[2]
                if var not in variables:
                    vl_report["dropped"].append((vlmoid, f"变量 {var} 不在本域变量")); continue
                entries=[]
                for item in sorted(VLM_DATA[vlmoid], key=lambda x: x.get("order",1e9)):
                    for c in WC_DATA.get(item.get("where"),[]):
                        if c.get("comparator") in ("EQ","IN"):
                            claimed[c["variable"]] |= {dnorm(v) for v in c["values"]}
                    reason=[]; e=make_vl_entry(item, variables[var].get("label"), xpt_df, reason)
                    if e: entries.append(e)
                    else: vl_report["dropped"].append((f"{vlmoid}.{item.get('item')}", reason[-1] if reason else "?"))
                if entries:
                    variables[var]["value_list"]=entries
                    vl_report["injected"].append((vlmoid, [c["where"][0]["values"][0] if c["where"] else "?" for c in entries]))
            for dv,cs in claimed.items():
                if dv not in xpt_df.columns: continue
                vc=xpt_df[dv].map(dnorm).value_counts()
                miss=[(v,int(n)) for v,n in vc.items() if v and v not in cs]
                if miss: vl_report["uncovered"].append((f"{DOMAIN}.{dv}", miss))
    n_vl_main=sum(1 for v in variables.values() if "value_list" in v)

    # ---- SUPP ----
    SUPPDOM=f"SUPP{DOMAIN}"
    has_supp = Path(SUPP_XPT).exists() or bool(supp_xlsx)
    nvlist=0
    if has_supp:
        def vl_entry(q, label, otype):
            x=supp_xlsx.get(q, {})
            e=OrderedDict()
            e["where"]=[OrderedDict([("variable","QNAM"),("comparator","EQ"),("values",[q])])]
            if label: e["label"]=label
            if x.get("codelist"): e["codelist"]=x["codelist"]
            if x.get("method"):   e["method"]=x["method"]
            if x.get("comment"):      e["comment"]=x["comment"]
            if x.get("comment_text"): e["comment_text"]=x["comment_text"]
            origin=OrderedDict(type=otype or x.get("origin_x"))
            if x.get("source"): origin["source"]=x["source"]
            pages=crf_of(SUPPDOM, q)
            if pages: origin["crf_pages"]=pages
            e["origin"]=Flow(origin) if not pages else origin
            e["mandatory"]=False
            return e

        supp_meta={}; vlist=[]
        if Path(SUPP_XPT).exists():
            import pyreadstat
            sdf, sm = pyreadstat.read_xport(SUPP_XPT)
            swid=getattr(sm,"variable_storage_width",{}) or {}
            for i,n in enumerate(sm.column_names):
                supp_meta[n]=dict(order=i+1, label=sm.column_labels[i] if sm.column_labels else None, width=swid.get(n))
            _supp_has_no_data = len(sdf) == 0   # xpt 存在但行数=0
            qn_xpt=[]
            if "QNAM" in sdf.columns:
                seen=set()
                for q in sdf["QNAM"]:
                    if q and q not in seen: seen.add(q); qn_xpt.append(q)
            for q in qn_xpt:
                sub=sdf[sdf["QNAM"]==q]
                lab=sub["QLABEL"].dropna().iloc[0] if "QLABEL" in sdf.columns and not sub["QLABEL"].dropna().empty else None
                qo =sub["QORIG"].dropna().iloc[0]  if "QORIG"  in sdf.columns and not sub["QORIG"].dropna().empty  else None
                vlist.append(vl_entry(q, lab, QORIG_MAP.get(str(qo).upper(), qo) if qo else None))
            only_x=set(supp_xlsx)-set(qn_xpt); only_d=set(qn_xpt)-set(supp_xlsx)
            if only_x: warn.append(f"qnam 仅见于 xlsx 不在数据: {sorted(only_x)}")
            if only_d: warn.append(f"qnam 仅见于数据不在 xlsx(缺语义): {sorted(only_d)}")
        else:
            for q,x in supp_xlsx.items():
                vlist.append(vl_entry(q, x.get("label_x"), x.get("origin_x")))
            warn.append(f"未找到 {SUPP_XPT}：10 标准列无 length、qnam 列表/label 回退 xlsx")
        nvlist=len(vlist)

        sr=next((r for r in toc[1:] if cell(r,ti,"Dataset")=="SUPP--"), None)
        supp_ds=OrderedDict(domain=SUPPDOM)
        lbl=str(cell(sr,ti,DESC_COL) or SUPP_DESC_FALLBACK)
        supp_ds["label"]=lbl.replace("[domain name]",DOMAIN)
        scls=cell(sr,ti,"Class"); supp_ds["class"]=CLASS_MAP.get(scls,scls)
        spur=cell(sr,ti,"Purpose"); supp_ds["purpose"]=PURPOSE_MAP.get(spur,spur) or DEFAULT_PURPOSE
        supp_ds["repeating"]=True
        supp_ds["is_reference_data"]=False
        supp_ds["structure"]=SUPP_STRUCTURE
        supp_ds["sas_dataset"]=SUPPDOM
        sk=cell(sr,ti,"Keys"); supp_ds["keys"]=[k.strip() for k in str(sk).split(",")] if sk else []
        loc=cell(sr,ti,"Location"); supp_ds["archive"]=str(loc).replace("supp--",SUPPDOM.lower()) if loc else f"{SUPPDOM.lower()}.xpt"
        # has_no_data: supp xpt 存在且行数=0（如 SUPPRP 空表）
        if Path(SUPP_XPT).exists() and _supp_has_no_data:
            supp_ds["has_no_data"] = True
            warn.append(f"{SUPPDOM}: xpt 行数=0 → has_no_data=True")

        supp_vars=OrderedDict()
        for i,(name,tmpl) in enumerate(SUPP_TEMPLATE):
            mta=supp_meta.get(name,{})
            v=OrderedDict()
            v["label"]=mta.get("label") or SUPP_LABELS.get(name)
            v["type"]="text"
            if mta.get("width") is not None: v["length"]=str(mta["width"])
            v["order"]=mta.get("order", i+1)
            v["mandatory"]=tmpl["mandatory"]
            if "key" in tmpl: v["key_sequence"]=tmpl["key"]
            o=OrderedDict(type=tmpl["origin"][0]); o["source"]=tmpl["origin"][1]; v["origin"]=Flow(o)
            if tmpl.get("codelist"): v["codelist"]=tmpl["codelist"]
            if tmpl.get("method"):   v["method"]=tmpl["method"]
            if tmpl.get("qval"):     v["value_list"]=vlist
            supp_vars[name]=v
        supp_ds["variables"]=supp_vars
        ds["supplemental"]=supp_ds

    # ---- 写出 ----
    out_path = Path(OUT_DIR) / OUT
    out_path.parent.mkdir(parents=True, exist_ok=True)
    src_note = XPT_PATH if not has_supp else f"{XPT_PATH}+{SUPP_XPT}"
    with open(out_path,"w",encoding="utf-8") as f:
        f.write(f"# 首版 {OUT}：{src_note}(结构) + xlsx(语义) + 部门值级库 + aCRF(页码) 合成。之后由 AI 维护。\n")
        yaml.dump(ds, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=100)
    return out_path, len(variables), nvlist, n_vl_main, has_supp, warn, vl_report

# ============================================================================
#  遍历 SDTM_DIR
# ============================================================================
mains={}; supps={}
for p in sorted(glob.glob(os.path.join(SDTM_DIR, "*.xpt"))):
    fn=os.path.basename(p).lower(); stem=fn[:-4] if fn.endswith(".xpt") else fn
    if stem.startswith("supp"): supps[stem[4:].upper()]=p
    else: mains[stem.upper()]=p
orphan=set(supps)-set(mains)

# ---------- 构建概览(头部) ----------
_BAR, _SUB = "═"*58, "─"*58
print("\n"+_BAR)
print("  🏗️  域 YAML 构建 (datasets/)")
print(_BAR)
print(f"  📂 {_pad('数据目录',10)}{SDTM_DIR}")
print(f"  📄 {_pad('aCRF',10)}{_n_acrf} 条 变量→页码" + ("" if _n_acrf else "  (无 acrf_pages.yaml)"))
if _vlm_found:
    print(f"  🧬 {_pad('值级库',10)}{len(VLM_DATA)} VLMOID · {len(WC_DATA)} where  (language={LANG})")
else:
    print(f"  🧬 {_pad('值级库',10)}未找到 → 跳过主域值级注入")
print(f"  🔍 {_pad('扫描',10)}主域 {len(mains)} 个 · supp {len(supps)} 个")
if orphan: print(f"  ⚠  孤儿 supp(无对应主域): {sorted(orphan)}")
print(_SUB)
print("  📦 逐域生成")

ok=[]; skip=[]; all_warn=[]; vl_all={}
tot_vars=tot_vl_main=tot_supp=n_with_supp=0
for DOMAIN in sorted(mains):
    XPT_PATH = mains[DOMAIN]
    SUPP_XPT = supps.get(DOMAIN, os.path.join(SDTM_DIR, f"supp{DOMAIN.lower()}.xpt"))
    out_path, nmain, nvl, n_vl_main, has_supp, warn, vl_report = build_domain(DOMAIN, XPT_PATH, SUPP_XPT)
    if out_path is None:
        skip.append((DOMAIN, warn[-1] if warn else "?")); continue
    ok.append(DOMAIN); vl_all[DOMAIN]=vl_report
    tot_vars+=nmain; tot_vl_main+=n_vl_main
    if has_supp: n_with_supp+=1; tot_supp+=nvl
    tag = (f" · 值级 {n_vl_main}" if n_vl_main else "") + (f" · SUPP {nvl} qnam" if has_supp else " · 无 SUPP")
    print(f"     ✓ {_pad(DOMAIN,8)}主变量 {nmain:>3}{tag}")
    if warn: all_warn += [f"{DOMAIN}: {w}" for w in warn]

print(_SUB)
print(f"  📊 成功 {len(ok)} 域 · 跳过 {len(skip)} 域 · 主变量 {tot_vars} · 值级变量 {tot_vl_main}")
print(f"     SUPP {n_with_supp} 域 / {tot_supp} qnam · 无 SUPP {len(ok)-n_with_supp} 域")
print(f"  ⏱  耗时 {time.time()-_t0:.1f} s")
if skip:
    print("  ⛔ 跳过:")
    for d,why in skip: print(f"       · {d}: {why}")
if all_warn:
    print(f"  ⚠  警告 {len(all_warn)} 条 (结构/aCRF/qnam 等):")
    for w in all_warn[:40]: print(f"       - {w}")
    if len(all_warn)>40: print(f"       … 其余 {len(all_warn)-40} 条略")
print(_BAR)

# ---- 值级注入报告 ----
print("\n"+_BAR)
print("  🧬 值级注入报告")
print(_BAR)
any_vl=False
for DOMAIN in ok:
    rep=vl_all.get(DOMAIN) or {}
    inj=rep.get("injected",[]); drp=rep.get("dropped",[]); unc=rep.get("uncovered",[])
    if not (inj or drp or unc): continue
    any_vl=True; print(f"  [{DOMAIN}]")
    for vlmoid,vals in inj: print(f"     ✓ 注入 {vlmoid}: {vals}")
    for key,why  in drp: print(f"     ✗ 丢弃 {key}: {why}")
    for dv,miss  in unc: print(f"     ⚠ 未覆盖 {dv} (数据有/模板无): " + ", ".join(f"{v}({n})" for v,n in miss))
if not any_vl: print("  (无主域值级注入)")
print(_BAR)

# ============================================================================
#  写出 datasets/toc.yaml —— 先按 Class 大类顺序，再按 TOC 行序排列
#  Class 顺序遵循 SDTM-MSG 3.4.2 规范（FDA 推荐顺序）
#  供 build_define 确定 ItemGroupDef 顺序；字段可供下游直接读取，无需再解析 xlsx
# ============================================================================
# MSG 规定的 Class 大类顺序（CLASS_MAP 的中文值为 key）
# 以 TOC 的英文 Class 原值为排序键（与显示语言无关）
CLASS_ORDER_EN = [
    "TRIAL DESIGN", "SPECIAL-PURPOSE", "INTERVENTIONS", "EVENTS",
    "FINDINGS", "FINDINGS ABOUT", "RELATIONSHIP", "STUDY REFERENCE",
]
def class_rank(cls_en):
    try: return CLASS_ORDER_EN.index(str(cls_en).strip().upper())
    except ValueError: return len(CLASS_ORDER_EN)   # 未知 class 排最后

ok_set = set(ok)
seen_toc = set(); toc_entries_raw = []

for r in toc[1:]:
    dom = cell(r, ti, "Dataset")
    if not dom or dom in seen_toc: continue
    seen_toc.add(dom)
    if dom not in ok_set: continue

    e = OrderedDict()
    e["dataset"] = dom
    for col in ("Description", "DescriptionCN", "Class", "SubClass", "SubClassN",
                "ParentClass", "ParentClassN", "Structure", "StructureCN",
                "Purpose", "Keys", "Repeating", "IsReferenceData",
                "Location", "Flag", "StandardOID", "IsNonStandard",
                "HasNoData", "Comment OID", "Order", "HR Notes"):
        v = cell(r, ti, col)
        if v is not None:
            e[col] = v
    toc_entries_raw.append(e)

# xpt 里有但 TOC 未列出的域追加到末尾（兜底，class 未知排最后）
for dom in ok:
    if dom not in seen_toc:
        toc_entries_raw.append(OrderedDict(dataset=dom))

# 排序键 = TOC 英文 Class 原值（语言无关）；显示用 entry_class_disp（CN→中文, EN→原样）
def entry_class_disp(e):
    raw_cls = e.get("Class", "")
    return CLASS_MAP.get(raw_cls, raw_cls)

toc_entries = sorted(toc_entries_raw, key=lambda e: class_rank(e.get("Class", "")))

toc_yaml_path = Path(OUT_DIR) / "toc.yaml"
with open(toc_yaml_path, "w", encoding="utf-8") as f:
    f.write(f"# 数据集顺序表：按 SDTM-MSG Class 大类顺序，Class 内按 TOC 行序。\n")
    f.write(f"# Class 顺序: {CLASS_ORDER_EN}\n")
    f.write(f"# 来源: {SPEC_XLSX} Sheet=TOC，仅含本次成功生成的 {len(toc_entries)} 个域，保留全部原始字段。\n")
    yaml.dump(toc_entries, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=120)

# 打印顺序预览（按 class 分组显示）
from itertools import groupby
print("\n"+_BAR)
print(f"  🗂️  数据集顺序表 toc.yaml — {len(toc_entries)} 域 (按 Class 排序)")
print(_BAR)
for cls, grp in groupby(toc_entries, key=entry_class_disp):
    doms = [e["dataset"] for e in grp]
    print(f"  {cls or '(未知)'}: {doms}")
print(_BAR)


══════════════════════════════════════════════════════════
  🏗️  域 YAML 构建 (datasets/)
══════════════════════════════════════════════════════════
  📂 数据目录  m5\datasets\tabulations\sdtm
  📄 aCRF      232 条 变量→页码
  🧬 值级库    16 VLMOID · 83 where  (language=CN)
  🔍 扫描      主域 34 个 · supp 21 个
──────────────────────────────────────────────────────────
  📦 逐域生成


     ✓ AE      主变量  40 · SUPP 12 qnam
     ✓ CE      主变量  15 · SUPP 1 qnam


     ✓ CM      主变量  27 · SUPP 28 qnam
     ✓ CV      主变量  21 · SUPP 4 qnam
     ✓ DC      主变量  27 · SUPP 11 qnam
     ✓ DD      主变量  12 · SUPP 1 qnam
     ✓ DM      主变量  26 · SUPP 11 qnam


     ✓ DS      主变量  13 · SUPP 11 qnam
     ✓ DV      主变量  16 · SUPP 3 qnam
     ✓ EC      主变量  26 · SUPP 10 qnam
     ✓ EG      主变量  23 · SUPP 2 qnam


     ✓ EX      主变量  21 · SUPP 8 qnam
     ✓ IE      主变量  11 · SUPP 1 qnam
     ✓ IS      主变量  22 · 无 SUPP


     ✓ LB      主变量  30 · 无 SUPP
     ✓ MB      主变量  26 · 无 SUPP
     ✓ MH      主变量  27 · SUPP 11 qnam
     ✓ MI      主变量  24 · 无 SUPP
     ✓ PC      主变量  25 · SUPP 1 qnam


     ✓ PE      主变量  18 · SUPP 3 qnam
     ✓ PR      主变量  20 · SUPP 10 qnam
     ✓ RP      主变量  15 · SUPP 0 qnam


     ✓ RS      主变量  26 · 值级 6 · SUPP 4 qnam
     ✓ SE      主变量  13 · 值级 2 · 无 SUPP
     ✓ SS      主变量  15 · 无 SUPP
     ✓ SV      主变量  14 · 无 SUPP
     ✓ TA      主变量   8 · 无 SUPP
     ✓ TE      主变量   6 · 无 SUPP
     ✓ TI      主变量   6 · 无 SUPP


     ✓ TR      主变量  29 · SUPP 2 qnam
     ✓ TS      主变量  11 · 值级 1 · 无 SUPP
     ✓ TU      主变量  25 · SUPP 2 qnam
     ✓ TV      主变量   9 · 无 SUPP
     ✓ VS      主变量  23 · 无 SUPP
──────────────────────────────────────────────────────────
  📊 成功 34 域 · 跳过 0 域 · 主变量 670 · 值级变量 9
     SUPP 21 域 / 136 qnam · 无 SUPP 13 域
  ⏱  耗时 3.6 s
  ⚠  警告 116 条 (结构/aCRF/qnam 等):
       - AE: SUBJID: Collected 但 aCRF(AE) 无页码
       - AE: qnam 仅见于 xlsx 不在数据: ['AEACNC', 'AEACNG', 'AERELC', 'AERELG', 'AESOSP1']
       - CE: SUBJID: Collected 但 aCRF(CE) 无页码
       - CE: CEENRTPT: Collected 但 aCRF(CE) 无页码
       - CM: SUBJID: Collected 但 aCRF(CM) 无页码
       - CM: CMENRTPT: Collected 但 aCRF(CM) 无页码
       - CM: qnam 仅见于 xlsx 不在数据: ['CMDECOD2', 'CMDECOD3', 'CMDSFRMO', 'CMGRPID1', 'CMINDC1', 'CMOUTO1', 'CMTLNSP', 'CMTRT1', 'CMTRT2']
       - CV: SUBJID: Collected 但 aCRF(CV) 无页码
       - CV: CVTESTCD: Collected 但 aCRF(CV) 无页码
       - CV: CVTEST: Collected 但 aCRF(CV) 无页码
       - CV: CVORRES: Collected 但 aCRF(CV) 无


══════════════════════════════════════════════════════════
  🗂️  数据集顺序表 toc.yaml — 34 域 (按 Class 排序)
══════════════════════════════════════════════════════════
  试验设计: ['TA', 'TE', 'TI', 'TS', 'TV']
  SPECIAL-PURPOSE: ['DM', 'DC', 'SE', 'SV']
  干预类: ['CM', 'EC', 'EX', 'PR']
  事件类: ['AE', 'CE', 'DS', 'DV', 'MH']
  发现类: ['CV', 'DD', 'EG', 'IE', 'IS', 'LB', 'MB', 'MI', 'PC', 'PE', 'RP', 'RS', 'SS', 'TR', 'TU', 'VS']
══════════════════════════════════════════════════════════


## Codelist from Data + File

In [10]:
# ============================================================================
#  build_codelists (消费 datasets/*.yaml + std 门控 + decoded/decode 版)
#  分工: datasets/*.yaml 决定绑定 · xpt 决定用到哪些值 · standard_codelist 给 NCI/中文名/decode

#  std 门控: 值按 codelist 名比对 value_cn/value_en/decode_cn/decode_en
#        命中有NCI→成员(nci_code); 命中无NCI→extended:true; 不命中→丢
#  decoded 码表: 每个 item 额外输出 decode = 命中 std 项的 value_cn(回退用值本身), 与参考版一致
# ============================================================================
import os, glob, time, yaml, pyreadstat, pandas as pd
from pathlib import Path
from collections import defaultdict, OrderedDict
_t0 = time.time()

# ---------- 配置（均来自 config cell） ----------
DATASETS_DIR = DATASETS_DIR          # output_files.datasets_dir
SDTM_DIR     = SDTM_DIR              # config.yaml: sdtm_dir
STDCL        = STDCL_YAML            # STUDY_DIR / "standard_codelist.yaml"
CT_YAML      = SDTM_CT_YAML          # DEPT_DIR / ct_sdtm_*.yaml (官方 CDISC CT，term key 全为英文)
OUT          = CODELISTS_YAML        # STUDY_DIR / "codelists.yaml"

norm = lambda s: str(s).strip().casefold()

# ---------- standard_codelist + 类型映射 ----------
std = yaml.safe_load(Path(STDCL).read_text(encoding="utf-8"))
# 官方 CT yaml：{码表短名: {nci_code, items: {英文提交值: term_nci_code}}}
ct  = yaml.safe_load(Path(CT_YAML).read_text(encoding="utf-8")) if Path(CT_YAML).exists() else {}
_idx_cache={}
def std_index(short):
    if short in _idx_cache: return _idx_cache[short]
    c=std.get(short); idx=None
    if c and "items" in c:
        idx={}
        for it in c["items"]:
            for col in ("value_cn","value_en","decode_en","decode_cn"):
                v=it.get(col)
                if v is not None: idx.setdefault(norm(v), it)
    _idx_cache[short]=idx; return idx
def cl_type_of(short):                              # 类型: std 标注, 否则 enumerated
    t2=std.get(short,{}).get("type")
    return t2 if t2 in ("enumerated","decoded") else "enumerated"
def is_external(short):
    return std.get(short, {}).get("type") == "external"

# ---------- where 求值 ----------
def apply_where(df, where, warn, ctx):
    mask = pd.Series(True, index=df.index)
    for cond in where or []:
        var=cond.get("variable"); comp=str(cond.get("comparator","EQ")).upper()
        vals=[str(x) for x in (cond.get("values") or [])]
        if var not in df.columns:
            warn.append(f"{ctx}: where 变量 {var} 不在数据列 → 该子集空"); return df.iloc[0:0]
        col=df[var].astype(str)
        if   comp in ("EQ","IN"):       m=col.isin(vals)
        elif comp in ("NE","NOTIN"):    m=~col.isin(vals)
        else:
            warn.append(f"{ctx}: 不支持的 comparator {comp} → 忽略该子句"); continue
        mask &= m
    return df[mask]

# ---------- xpt 读取(缓存，按需只读 codelist 绑定列) ----------
_xpt_cache={}
def load_xpt(path, usecols=None):
    key=(path, tuple(sorted(usecols)) if usecols else None)
    if key not in _xpt_cache:
        try: _xpt_cache[key]=pyreadstat.read_xport(path, usecols=list(usecols) if usecols else None)[0]
        except Exception as e: _xpt_cache[key]=e
    return _xpt_cache[key]

# ---------- 遍历 datasets/*.yaml 收集 (码表 → 值集) ----------
cl_vals=defaultdict(set); ext_used=set(); warn=[]; n_ref=0
def register(cl, vals):
    if not cl: return
    if is_external(cl): ext_used.add(cl); return
    cl_vals[cl].update(str(v) for v in vals if str(v).strip()!="")

def collect_dataset(dsdoc):
    global n_ref
    archive=os.path.basename(str(dsdoc.get("archive","")))
    xpt_path=os.path.join(SDTM_DIR, archive) if archive else None
    dom=dsdoc.get("domain","?")
    # 只读取有 codelist 绑定的列 + where 条件涉及的列，避免全量加载大文件
    needed_cols=set()
    for var, vdef in (dsdoc.get("variables") or {}).items():
        if not isinstance(vdef, dict): continue
        if vdef.get("codelist"): needed_cols.add(var)
        for entry in (vdef.get("value_list") or []):
            if entry.get("codelist"):
                needed_cols.add(var)
                for cond in (entry.get("where") or []): needed_cols.add(cond["variable"])
    df=load_xpt(xpt_path, usecols=needed_cols if needed_cols else None) if xpt_path else None
    if isinstance(df, Exception) or df is None:
        warn.append(f"{dom}: 读不到 xpt {archive!r}({df}) → 该域码表值取不到"); df=None
    for var, vdef in (dsdoc.get("variables") or {}).items():
        if not isinstance(vdef, dict): continue
        cl=vdef.get("codelist")
        if cl:
            n_ref+=1
            vals=[v for v in df[var].dropna().unique()] if (df is not None and var in df.columns) else []
            register(cl, vals)
            if cl and not is_external(cl) and df is not None and var not in df.columns:
                warn.append(f"{dom}.{var} → {cl}: 变量不在数据列")
        for entry in (vdef.get("value_list") or []):
            cle=entry.get("codelist")
            if not cle: continue
            n_ref+=1
            if df is None: register(cle, []); continue
            sub=apply_where(df, entry.get("where"), warn, f"{dom}.{var}->{cle}")
            vals=[v for v in sub[var].dropna().unique()] if var in sub.columns else []
            register(cle, vals)

# toc.yaml 已在 datasets/ 下，glob 时会读到，需排除
ds_files=sorted(f for f in glob.glob(os.path.join(DATASETS_DIR, "*.yaml"))
                if os.path.basename(f) != "toc.yaml")
seen_dom=set()
for fp in ds_files:
    doc=yaml.safe_load(Path(fp).read_text(encoding="utf-8"))
    if not doc: continue
    seen_dom.add(doc.get("domain"))
    collect_dataset(doc)
    if isinstance(doc.get("supplemental"), dict):
        collect_dataset(doc["supplemental"])

xpt_doms={os.path.basename(p)[:-4].upper() for p in glob.glob(os.path.join(SDTM_DIR,"*.xpt"))
          if not os.path.basename(p).lower().startswith("supp")}
no_yaml=sorted(xpt_doms - {str(d).upper() for d in seen_dom})

# ---------- 组装输出(std 门控 + decoded 加 decode) ----------
out=OrderedDict(); drop_all=[]; drop_part={}; no_std=[]
def ct_term_nci(short, it):
    """从官方 CT yaml 补充 term 级 NCI Code。
    CN/EN 两模式逻辑一致：CT 的 key 全为英文，统一经 standard_codelist 的
    value_en 字段桥接（std.value_en 即中/英文提交值通向 CT 的桥）。
    """
    ct_items = ct.get(short, {}).get("items", {})
    en_val = it.get("value_en")
    return ct_items.get(en_val) if (ct_items and en_val) else None

def ct_cl_nci(short):
    """从官方 CT yaml 补充码表级 NCI Code。"""
    return ct.get(short, {}).get("nci_code")

def items_of(short, vals, decoded):
    """门控: 命中+nci→成员; 命中无nci→CT兜底→extended; 不命中→丢。
    decoded 时 item 加 decode=value_cn(回退值本身)。
    NCI Code 优先级: standard_codelist.CTAlias > 官方CT yaml（CN需经value_en桥接）
    """
    idx=std_index(short)
    if idx is None:                                  # std 无定义 → 全留 extended
        items=[]
        for v in sorted(vals):
            if str(v).strip()=="": continue
            e=OrderedDict(value=v)
            if decoded: e["decode"]=v
            # EN模式可直接查 CT yaml
            if LANG == "EN":
                nci = ct.get(short, {}).get("items", {}).get(v)
                if nci: e["nci_code"] = nci
                else:   e["extended"] = True
            else:
                e["extended"]=True               # CN模式无 std 映射，无法桥接
            items.append(e)
        return items, [], True
    items=[]; dropped=[]
    for v in sorted(vals):
        if str(v).strip()=="": continue
        it=idx.get(norm(v))
        if it is None: dropped.append(v); continue
        e=OrderedDict(value=v)
        if decoded: e["decode"]=it.get("value"+SUF) or v
        if it.get("nci_code"):
            e["nci_code"]=it["nci_code"]             # 第一优先: standard_codelist.CTAlias
        else:
            nci = ct_term_nci(short, it)             # 第二优先: 官方CT yaml（CN经value_en桥接）
            if nci: e["nci_code"]=nci
            else:   e["extended"]=True
        items.append(e)
    return items, dropped, False

for cl, vals in cl_vals.items():
    c=std.get(cl); e=OrderedDict()
    nm = c.get("name"+SUF) if c else None
    if nm: e["name"]=nm
    e["data_type"]="text"
    if c and c.get("nci_code"):
        e["nci_code"]=c["nci_code"]                  # 第一优先: standard_codelist.ParentAlias
    else:
        nci = ct_cl_nci(cl)                          # 第二优先: 官方CT yaml 码表级 NCI
        if nci: e["nci_code"]=nci
    typ=cl_type_of(cl); decoded=(typ=="decoded")
    items, dropped, missing = items_of(cl, vals, decoded)
    if missing: no_std.append((cl, len(items)))
    if not items:
        drop_all.append((cl, len(vals))); continue
    if dropped: drop_part[cl]=dropped
    e["type"]=typ; e["items"]=items; out[cl]=e
for cl in ext_used:
    c=std.get(cl, {}); e=OrderedDict()
    nm=c.get("name"+SUF)
    if nm: e["name"]=nm
    e["data_type"]="text"; e["type"]="external"; e["dictionary"]=c.get("dictionary")
    if c.get("version"): e["version"]=str(c["version"])
    if c.get("href"): e["href"]=c["href"]
    out[cl]=e

out=OrderedDict(sorted(out.items()))
with open(OUT,"w",encoding="utf-8") as f:
    f.write("# 数据优先 codelists.yaml: 绑定←datasets/*.yaml · 值←xpt · 取舍←standard_codelist门控 · 类型←std.type · decoded含decode. 字母序.\n")
    yaml.dump(out, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=120)

_enum  = sum(c.get("type")=="enumerated" for c in out.values())
_dec   = sum(c.get("type")=="decoded"    for c in out.values())
_ext   = sum(c.get("type")=="external"   for c in out.values())
_items = [it for c in out.values() for it in c.get("items",[])]
_n_item= len(_items)
_it_nci= sum(1 for it in _items if it.get("nci_code"))
_it_ext= sum(1 for it in _items if it.get("extended"))
_cl_nci= sum(1 for c in out.values() if c.get("nci_code"))
_ct_note = ("CN: 经 value_en 桥接" if LANG=="CN" else "EN: 直接匹配") if ct else "未加载"

def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print(); print(_BAR)
print("  🏷️  Codelists 组装 (codelists.yaml)")
print(_BAR)
print(f"  📥 {_pad('输入',6)}datasets/ {len(ds_files)} 域 · std 码表 {len(std)} · 官方CT {len(ct)} ({_ct_note})")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s")
print(_SUB)
print(f"  📊 码表 {len(out)} 个 (枚举 {_enum} · decoded {_dec} · 外部 {_ext})   ·   引用 {n_ref} 处   ·   项 {_n_item} 个")
print(f"  🔖 NCI 编码覆盖")
print(f"       · 项有 nci_code        {_it_nci:>4} / {_n_item}")
print(f"       · 项 extended(无 NCI)  {_it_ext:>4} / {_n_item}")
print(f"       · 码表级有 nci_code     {_cl_nci:>4} / {_enum+_dec}")
if no_yaml or no_std or drop_all or drop_part or warn:
    print(_SUB)
    print("  ⚠ 质量提示")
    if no_yaml:  print(f"       · 缺 yaml(有 xpt 无域): {no_yaml}")
    if no_std:   print(f"       · std 无定义(全留 extended) {len(no_std)} 个: {[c for c,_ in no_std]}")
    if drop_all: print(f"       · 整表丢弃 {len(drop_all)} 个: " + ", ".join(f"{c}({n})" for c,n in sorted(drop_all,key=lambda x:-x[1])))
    if drop_part:
        print(f"       · 部分丢弃 {len(drop_part)} 个码表(审是否有合法值漏勾 X):")
        for cl,dv in sorted(drop_part.items(), key=lambda x:-len(x[1])):
            print(f"           {cl:14s} 丢 {len(dv):4d}  例: {[str(v)[:18] for v in dv[:3]]}")
    if warn:
        print(f"       · WARN {len(warn)} 条:")
        for w in warn[:15]: print(f"           - {w}")
        if len(warn)>15: print(f"           … 其余 {len(warn)-15} 条略")
print(_BAR)


══════════════════════════════════════════════════════════
  🏷️  Codelists 组装 (codelists.yaml)
══════════════════════════════════════════════════════════
  📥 输入  datasets/ 34 域 · std 码表 174 · 官方CT 1145 (CN: 经 value_en 桥接)
  📤 输出  study_std\codelists.yaml   (48.6 KB)
  ⏱  耗时  5.4 s
──────────────────────────────────────────────────────────
  📊 码表 145 个 (枚举 142 · decoded 0 · 外部 3)   ·   引用 364 处   ·   项 765 个
  🔖 NCI 编码覆盖
       · 项有 nci_code         429 / 765
       · 项 extended(无 NCI)   336 / 765
       · 码表级有 nci_code       90 / 142
──────────────────────────────────────────────────────────
  ⚠ 质量提示
       · std 无定义(全留 extended) 1 个: ['TPHASE']
       · 整表丢弃 4 个: PCTPTNUM(3), ENTPT(2), ECOG101OR(2), IS_UNIT(0)
       · 部分丢弃 17 个码表(审是否有合法值漏勾 X):
           TRPROPRS       丢 1228  例: ['-0.1', '-0.2', '-0.3']
           CMINDC         丢  746  例: ['Abdominal pain', 'Acid inhibition an', 'Acid suppression']
           PEORRES        丢  186  例: ['2024-6-27  appeara', 'A 10 cm long trans', '

## Methods

In [11]:
# ============================================================================
#  生成项目级 methods.yaml: 遍历 datasets/*.yaml 收集 method 引用(三处来源),
#  在部门方法库取子集, 按 config.language 收敛单语。
#  来源: 1 变量级 variables[VAR].method  2 值级 value_list[].method  3 supplemental 递归
#  注意: 需先跑 build_domain(已注入 value_list)再跑本 cell, 否则收不到值级方法。
#  依赖: pyyaml
# ============================================================================
import yaml, glob, os, time
from pathlib import Path
from collections import OrderedDict, defaultdict
_t0 = time.time()

# ---------- 配置(均来自 config cell) ----------
LIB      = METHODS_YAML       # DEPT_DIR / (methods_xlsx stem + ".yaml")
DATA_DIR = DATASETS_DIR       # output_files.datasets_dir
OUT      = OUT_METHODS_YAML   # STUDY_DIR / "methods.yaml"
# LANG / SUF 直接沿用 config cell 已暴露的同名变量

lib=yaml.safe_load(open(LIB,encoding="utf-8")) or {}

# 收集一个 domain-like dict 的 method 引用 → refs[method]=[来源], 并标记值级
def collect(node, dom, refs, vlevel):
    if not isinstance(node, dict): return
    for var, vd in (node.get("variables") or {}).items():
        if not isinstance(vd, dict): continue
        if vd.get("method"):                                  # 1 变量级
            refs[vd["method"]].append(f"{dom}.{var}")
        for e in (vd.get("value_list") or []):                # 2 值级
            if isinstance(e, dict) and e.get("method"):
                w=e.get("where")
                wtag=w[0]["values"][0] if isinstance(w,list) and w and w[0].get("values") else "?"
                refs[e["method"]].append(f"{dom}.{var}[{wtag}]"); vlevel.add(e["method"])
    sup=node.get("supplemental")                              # 3 SUPP 递归
    if isinstance(sup, dict):
        collect(sup, sup.get("domain", dom+"·SUPP"), refs, vlevel)

refs=defaultdict(list); vlevel=set()
# 排除 toc.yaml,只遍历域 yaml
files=sorted(f for f in glob.glob(os.path.join(DATA_DIR,"*.yaml"))
             if os.path.basename(f) != "toc.yaml")
for fp in files:
    d=yaml.safe_load(open(fp,encoding="utf-8")) or {}
    collect(d, d.get("domain", Path(fp).stem.upper()), refs, vlevel)

used   =sorted(k for k in refs if k in lib)
missing=sorted(k for k in refs if k not in lib)
unused =sorted(k for k in lib if k not in refs)

def resolve(m):
    o=OrderedDict()
    nm=m.get("name"+SUF) or m.get("name_en") or m.get("name_cn")
    if nm: o["name"]=nm
    if m.get("type"): o["type"]=m["type"]
    de=m.get("description"+SUF) or m.get("description_en") or m.get("description_cn")
    if de: o["description"]=de
    return o

out=OrderedDict((k,resolve(lib[k])) for k in used)
with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# 项目级 methods (language={LANG}, 从 {len(files)} 个 domain yaml 收集变量级+值级引用, 在 {LIB} 中取子集)。\n")
    yaml.dump(out,f,allow_unicode=True,default_flow_style=False,sort_keys=False,width=200)

# ---------- 统计 ----------
vl_used = sorted(k for k in used if k in vlevel)
n_sites = sum(len(v) for v in refs.values())             # 总引用点数(含重复使用)
_top    = sorted(((k,len(refs[k])) for k in used), key=lambda kv: kv[1], reverse=True)[:5]

def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  🔧 项目级 Methods 收集 (methods.yaml)")
print(_BAR)
print(f"  📥 {_pad('输入',6)}方法库 {len(lib)} 条 · datasets/ {len(files)} 域")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s")
print(_SUB)
print(f"  📊 命中 {len(used)} / 库 {len(lib)}   ·   引用点 {n_sites}   ·   未用 {len(unused)} · 未解析 {len(missing)}")
if vl_used:
    print(f"  🧬 其中值级方法 {len(vl_used)}: {vl_used}")
if _top:
    print(f"  🏆 最常引用 (top{len(_top)})")
    for k,n in _top: print(f"       · {_pad(k,16)}{n:>4} 处")
if missing or unused:
    print(_SUB)
    print("  ⚠ 质量提示")
    if missing: print(f"       · 未解析(引用了但库无，需补库/核命名) {len(missing)} 个: {missing}")
    if unused:  print(f"       · 未使用(库里无人引用) {len(unused)} 个: {unused}")
print(_BAR)


══════════════════════════════════════════════════════════
  🔧 项目级 Methods 收集 (methods.yaml)
══════════════════════════════════════════════════════════
  📥 输入  方法库 49 条 · datasets/ 34 域
  📤 输出  study_std\methods.yaml   (5.2 KB)
  ⏱  耗时  0.5 s
──────────────────────────────────────────────────────────
  📊 命中 33 / 库 49   ·   引用点 207   ·   未用 16 · 未解析 2
  🧬 其中值级方法 6: ['AETRTEM', 'SEENDTC.FU', 'SEENDTC.SCRN', 'SESTDTC.FU', 'SESTDTC.SCRN', 'STRESC.STD']
  🏆 最常引用 (top5)
       · USUBJID           50 处
       · DY                35 处
       · SEQ               28 处
       · EPOCH             19 处
       · STRESC            13 处
──────────────────────────────────────────────────────────
  ⚠ 质量提示
       · 未解析(引用了但库无，需补库/核命名) 2 个: ['EXENDY', 'EXSTDY']
       · 未使用(库里无人引用) 16 个: ['APID', 'ARMNRS', 'BLFL', 'EGSTRESC.QTCF', 'IEORRES', 'RDEVID', 'RFCENDTC', 'RFCSTDTC', 'RSUBJID', 'SEENDTC.TRT', 'SESTDTC.TRT', 'STNRHI', 'STNRLO', 'STTPT', 'SVSTDTC', 'TRT']
═══════════════════════════════════════════

## Comments

In [12]:
# ============================================================================
#  生成项目级 comments.yaml(两层) —— 全部来自 datasets/*.yaml(domain yaml 即唯一真源)
#    第一层 = comment(key)能在部门标准库查到 → 文本走库, 保留 doc/page
#    第二层 = 库里查不到, 但 domain yaml 自带 comment_text(研究专属正文, build_domain 从 spec Study Derivation 抓的)
#  按 config.language 收敛单语。来源: 数据集级 · 变量级 · value_list 级 · supplemental 递归
#  依赖: pyyaml
# ============================================================================
import yaml, glob, os, time
from pathlib import Path
from collections import OrderedDict, defaultdict
_t0 = time.time()

# ---------- 配置（均来自 config cell） ----------
LIB      = COMMENTS_LIB_YAML   # DEPT_DIR / (comments_xlsx stem + ".yaml")
DATA_DIR = DATASETS_DIR         # output_files.datasets_dir
OUT      = OUT_COMMENTS_YAML    # STUDY_DIR / "comments.yaml"
# LANG / SUF 直接沿用 config cell 已暴露的同名变量

lib=yaml.safe_load(open(LIB,encoding="utf-8")) or {}

# 收集 domain yaml 的 comment 引用 + 自带正文
def collect(node, dom, refs, study_text):
    if not isinstance(node, dict): return
    def take(key, text, src):
        refs[key].append(src)
        if text: study_text.setdefault(key, text)             # 自带研究专属正文(首次为准)
    if node.get("comment"): take(node["comment"], node.get("comment_text"), dom)   # 数据集级
    for var, vd in (node.get("variables") or {}).items():
        if not isinstance(vd, dict): continue
        if vd.get("comment"): take(vd["comment"], vd.get("comment_text"), f"{dom}.{var}")  # 变量级
        for e in (vd.get("value_list") or []):                                              # 值级
            if isinstance(e, dict) and e.get("comment"):
                w=e.get("where")
                wtag=w[0]["values"][0] if isinstance(w,list) and w and w[0].get("values") else "?"
                take(e["comment"], e.get("comment_text"), f"{dom}.{var}[{wtag}]")
    sup=node.get("supplemental")
    if isinstance(sup, dict): collect(sup, sup.get("domain", dom+"·SUPP"), refs, study_text)

refs=defaultdict(list); study_text={}
# 排除 toc.yaml，只遍历域 yaml
files=sorted(f for f in glob.glob(os.path.join(DATA_DIR,"*.yaml"))
             if os.path.basename(f) != "toc.yaml")
for fp in files:
    d=yaml.safe_load(open(fp,encoding="utf-8")) or {}
    collect(d, d.get("domain", Path(fp).stem.upper()), refs, study_text)

# ---- 两层解析(全部基于 domain yaml) ----
out=OrderedDict(); L1=[]; L2=[]; missing=[]
for k in sorted(refs):
    if k in lib:                                              # 第一层: 部门库
        m=lib[k]; o=OrderedDict()
        de=m.get("description"+SUF) or m.get("description_en") or m.get("description_cn")
        if de: o["description"]=de
        for kk in ("doc_id","page_ref_type","page_refs"):
            if m.get(kk): o[kk]=m[kk]
        out[k]=o; L1.append(k)
    elif k in study_text:                                     # 第二层: domain yaml 自带正文
        out[k]=OrderedDict(description=study_text[k]); L2.append(k)
    else:
        missing.append(k)

with open(OUT,"w",encoding="utf-8") as f:
    f.write(f"# 项目级 comments (language={LANG}): 第一层={len(L1)} 部门库 + 第二层={len(L2)} 研究专属(domain yaml 自带 comment_text)。\n")
    yaml.dump(out,f,allow_unicode=True,default_flow_style=False,sort_keys=False,width=200)

# ---------- 统计 ----------
n_sites = sum(len(v) for v in refs.values())             # 总引用点数(含重复使用)
_top    = sorted(((k,len(refs[k])) for k in out), key=lambda kv: kv[1], reverse=True)[:5]

def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58

print("\n"+_BAR)
print("  💬 项目级 Comments 收集 (comments.yaml)")
print(_BAR)
print(f"  📥 {_pad('输入',6)}注释库 {len(lib)} 条 · datasets/ {len(files)} 域")
print(f"  📤 {_pad('输出',6)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',6)}{time.time()-_t0:.1f} s")
print(_SUB)
print(f"  📊 共 {len(out)} 条  =  第一层(部门库) {len(L1)}  +  第二层(研究专属) {len(L2)}   ·   引用点 {n_sites}")
if _top:
    print(f"  🏆 最常引用 (top{len(_top)})")
    for k,n in _top: print(f"       · {_pad(str(k),16)}{n:>4} 处")
if missing:
    print(_SUB)
    print(f"  ⚠ 未解析 {len(missing)} 条(库无 & domain yaml 也无 comment_text，需补库或补 Study Derivation)")
    for k in missing:
        print(f"       · {k}  ←  {', '.join(refs[k][:5])}{' …' if len(refs[k])>5 else ''}")
print(_BAR)


══════════════════════════════════════════════════════════
  💬 项目级 Comments 收集 (comments.yaml)
══════════════════════════════════════════════════════════
  📥 输入  注释库 19 条 · datasets/ 34 域
  📤 输出  study_std\comments.yaml   (3.3 KB)
  ⏱  耗时  0.6 s
──────────────────────────────────────────────────────────
  📊 共 55 条  =  第一层(部门库) 12  +  第二层(研究专属) 43   ·   引用点 139
  🏆 最常引用 (top5)
       · MEDDRA            24 处
       · VISIT             18 处
       · VISITNUM          18 处
       · SPID              12 处
       · ENRTPT             5 处
──────────────────────────────────────────────────────────
  ⚠ 未解析 5 条(库无 & domain yaml 也无 comment_text，需补库或补 Study Derivation)
       · ENTPT  ←  CE.CEENTPT
       · NC  ←  RP.RPORRES, SE.SESTDTC, SE.SEENDTC, TU.TUMETHOD
       · SS.VISITDY  ←  SS.VISITDY
       · STAT  ←  CV.CVSTAT
       · TV.ARM  ←  TV.ARM
══════════════════════════════════════════════════════════


## Study Info

In [13]:
# ============================================================================
#  build_study_yaml —— 生成 study_std/study.yaml（项目元数据，手动维护部分用占位符）
#  大部分字段从 config cell 已有变量推导，少数需在下方手动填写。
#  生成后请核对 study_description / originator / metadata_version_oid 等字段。
#  依赖: pyyaml
# ============================================================================
import yaml, re, os, time
from pathlib import Path
from collections import OrderedDict
_t0 = time.time()

# ---------- 从已有变量推导 ----------
# STUDY_ID         → "SHR-1234-III-301"  (config cell: 从 spec 文件名推导)
# SDTM_CT_XLSX     → "SDTM_CT_2024-09-27.xlsx"  (config cell: DEPT_DIR / ...)
# STUDY_DIR        → Path("study_std")  (config cell)
# XML_LANG         → "zh" | "en"  (config cell)

# CT 版本从文件名提取: "SDTM_CT_2024-09-27.xlsx" → "2024-09-27"
_ct_stem = Path(SDTM_CT_XLSX).stem          # SDTM_CT_2024-09-27
_ct_ver  = re.search(r'\d{4}-\d{2}-\d{2}', _ct_stem)
CT_VERSION = _ct_ver.group() if _ct_ver else "YYYY-MM-DD"

# ---------- 语言相关文案（EN 模式不得出现中文） ----------
MDV_LABEL    = f"{STUDY_ID} SDTM数据说明" if LANG=="CN" else f"{STUDY_ID} SDTM Metadata"
ACRF_TITLE   = "注释病例报告表"         if LANG=="CN" else "Annotated Case Report Form"
CALGO_TITLE  = "复杂算法"               if LANG=="CN" else "Complex Algorithms"
RGUIDE_TITLE = "临床研究数据审阅说明"   if LANG=="CN" else "Clinical Study Data Reviewer's Guide"

# ---------- 需要手动填写的字段（请在此处修改后重新运行） ----------
# 占位符格式: {{FIELD_NAME}}，方便 LLM 批量识别和替换
# STUDY_ID 已由 config cell 从 spec 文件名自动推导，如需覆盖可在此处取消注释：
# STUDY_ID = "{{STUDY_ID}}"

STUDY_DESCRIPTION  = "三期肿瘤对照试验"   # 研究中文全称
ORIGINATOR         = "{{ORIGINATOR}}"           # 申办方名称（公司全称）
SDTMIG_VERSION     = "3.3"                      # SDTM-IG 版本
SDTM_VERSION       = "1.7"                      # SDTM 版本
DEFINE_VERSION     = "2.1.0"                    # Define-XML 版本
ODM_VERSION        = "1.3.2"                    # ODM 版本
MEDDRA_VERSION     = "{{MEDDRA_VERSION}}"       # 如 28.0
WHODRUG_VERSION    = "{{WHODRUG_VERSION}}"       # 如 GLOBALB3Mar23

# ---------- 构建 ----------

study = OrderedDict([
    ("study", OrderedDict([
        ("oid",         STUDY_ID),
        ("name",        STUDY_ID),
        ("description", STUDY_DESCRIPTION),
        ("protocol",    STUDY_ID),
    ])),
    ("file", OrderedDict([
        ("originator",   ORIGINATOR),
        ("context",      "Submission"),
        ("odm_version",  ODM_VERSION),
    ])),
    ("metadata_version", OrderedDict([
        ("oid",            f"{STUDY_ID}.SDTMIG.{SDTMIG_VERSION}.SDTM.{SDTM_VERSION}"),
        ("name",           MDV_LABEL),
        ("description",    MDV_LABEL),
        ("define_version", DEFINE_VERSION),
    ])),
    ("standards", [
        OrderedDict([("oid","SDTM-IG"), ("name","SDTMIG"),    ("type","IG"),
                     ("version", SDTMIG_VERSION), ("status","Final")]),
        OrderedDict([("oid","SDTM-CT"), ("name","CDISC/NCI"), ("type","CT"),
                     ("version", CT_VERSION),     ("status","Final"), ("publishing_set","SDTM")]),
    ]),
    ("documents", OrderedDict([
        ("annotated_crf", OrderedDict([
            ("leaf_id", "Annotated CRF"),
            ("href",    "acrf.pdf"),
            ("title",   ACRF_TITLE),
        ])),
        ("supplemental", [
            OrderedDict([("leaf_id","ComplexAlgorithms"), ("href","complexalgorithms.pdf"), ("title",CALGO_TITLE)]),
            OrderedDict([("leaf_id","ReviewerGuide"),     ("href","csdrg.pdf"),             ("title",RGUIDE_TITLE)]),
        ]),
    ])),
    ("dictionaries", [
        OrderedDict([("codelist","CTCAE"),    ("dictionary","CTCAE"),                        ("version","5.0")]),
        OrderedDict([("codelist","DRUGDICT"),  ("dictionary","WHODRUG"),                     ("version", WHODRUG_VERSION)]),
        OrderedDict([("codelist","ISO21090"),  ("dictionary","ISO 21090 NullFlavor enumeration"), ("version","2011-02")]),
        OrderedDict([("codelist","ISO3166"),   ("dictionary","ISO 3166-1 Alpha-3"),           ("version","2013-11-15")]),
        OrderedDict([("codelist","MEDDRACT"),  ("dictionary","MEDDRA"),                      ("version", MEDDRA_VERSION)]),
    ]),
])

OUT = STUDY_DIR / "study.yaml"
with open(OUT, "w", encoding="utf-8") as f:
    f.write(f"# study.yaml —— 项目元数据，由 build_study_yaml cell 生成，之后手动维护。\n")
    f.write(f"# 生成时间: {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"# 研究号: {STUDY_ID}  CT版本: {CT_VERSION}  SDTM-IG: {SDTMIG_VERSION}\n")
    yaml.dump(study, f, allow_unicode=True, default_flow_style=False, sort_keys=False, width=120)

# ---------- 打印确认 ----------
def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58
_stds  = study["standards"]; _dicts = study["dictionaries"]
_supp  = study["documents"]["supplemental"]

print("\n"+_BAR)
print("  📋 项目元数据 (study.yaml)")
print(_BAR)
print(f"  📤 {_pad('输出',8)}{OUT}   ({_sz(OUT)})")
print(f"  ⏱  {_pad('耗时',8)}{time.time()-_t0:.1f} s")
print(_SUB)
print(f"  🔖 {_pad('研究号',8)}{STUDY_ID}   ·   语言 xml:lang={XML_LANG}")
print(f"  🧬 {_pad('版本',8)}Define {DEFINE_VERSION} · SDTM-IG {SDTMIG_VERSION} · SDTM {SDTM_VERSION} · ODM {ODM_VERSION} · CT {CT_VERSION}")
print(f"  🔑 {_pad('MDV OID',8)}{study['metadata_version']['oid']}")
print(f"  📚 {_pad('标准',8)}{len(_stds)} 项: " + " · ".join(s["name"] for s in _stds))
print(f"  📖 {_pad('字典',8)}{len(_dicts)} 个: " + " · ".join(f"{d['dictionary']} {d['version']}" for d in _dicts))
print(f"  📄 {_pad('文档',8)}aCRF + 补充 {len(_supp)} 个")

# 提示需要手动核对的字段
todos = [(k,v) for k,v in [
    ("study.description",    STUDY_DESCRIPTION),
    ("file.originator",      ORIGINATOR),
    ("dictionaries.MEDDRA",  MEDDRA_VERSION),
    ("dictionaries.WHODRUG", WHODRUG_VERSION),
] if "{{" in str(v)]
if todos:
    print(_SUB)
    print(f"  ⚠ 待填写 {len(todos)} 个占位符（在 cell 顶部填好后重新运行）")
    for k,v in todos: print(f"       · {_pad(k,22)}{v}")
print(_BAR)


══════════════════════════════════════════════════════════
  📋 项目元数据 (study.yaml)
══════════════════════════════════════════════════════════
  📤 输出    study_std\study.yaml   (1.5 KB)
  ⏱  耗时    0.0 s
──────────────────────────────────────────────────────────
  🔖 研究号  SHR-1234-III-301   ·   语言 xml:lang=zh
  🧬 版本    Define 2.1.0 · SDTM-IG 3.3 · SDTM 1.7 · ODM 1.3.2 · CT 2024-09-27
  🔑 MDV OID SHR-1234-III-301.SDTMIG.3.3.SDTM.1.7
  📚 标准    2 项: SDTMIG · CDISC/NCI
  📖 字典    5 个: CTCAE 5.0 · WHODRUG {{WHODRUG_VERSION}} · ISO 21090 NullFlavor enumeration 2011-02 · ISO 3166-1 Alpha-3 2013-11-15 · MEDDRA {{MEDDRA_VERSION}}
  📄 文档    aCRF + 补充 2 个
──────────────────────────────────────────────────────────
  ⚠ 待填写 3 个占位符（在 cell 顶部填好后重新运行）
       · file.originator       {{ORIGINATOR}}
       · dictionaries.MEDDRA   {{MEDDRA_VERSION}}
       · dictionaries.WHODRUG  {{WHODRUG_VERSION}}
══════════════════════════════════════════════════════════


# 组装 define.xml

In [14]:
# ============================================================================
#  build_define —— 从 YAML 源确定性编译出 Define-XML 2.1
#  数据集顺序由 datasets/toc.yaml 决定（与 TOC sheet 行序一致）
#  依赖：pyyaml
# ============================================================================
import yaml, os, time, xml.etree.ElementTree as ET
from pathlib import Path
_t0 = time.time()

# ---------- 配置（均来自 config cell） ----------
STUDY_YAML   = STUDY_DIR / "study.yaml"  # STUDY_DIR（build_study_yaml 生成）
CODELISTS    = CODELISTS_YAML            # STUDY_DIR / "codelists.yaml"
METHODS      = OUT_METHODS_YAML          # STUDY_DIR / "methods.yaml"
COMMENTS     = OUT_COMMENTS_YAML         # STUDY_DIR / "comments.yaml"
DS_DIR       = Path(DATASETS_DIR)        # datasets/
TOC          = TOC_YAML                  # datasets/toc.yaml
OUT_FILE     = Path(DEFINE_XML)          # 工作目录 / "define.xml"
# STD_IG / STD_CT 在读取 study.yaml 后自动推导（见下方）

# ---------- 命名空间 ----------
ODM="http://www.cdisc.org/ns/odm/v1.3"; DEF="http://www.cdisc.org/ns/def/v2.1"
XLINK="http://www.w3.org/1999/xlink";   XML="http://www.w3.org/XML/1998/namespace"
ET.register_namespace("", ODM); ET.register_namespace("def", DEF); ET.register_namespace("xlink", XLINK)
def o(t): return f"{{{ODM}}}{t}"
def d(t): return f"{{{DEF}}}{t}"
def SE(parent, tag, text=None, **attrs):
    e=ET.SubElement(parent, tag, {k:str(v) for k,v in attrs.items() if v is not None})
    if text is not None: e.text=str(text)
    return e
def TT(parent, text, lang=XML_LANG):               # Description>TranslatedText，语言跟 config
    if text is None: return
    desc=SE(parent, o("Description")); SE(desc, o("TranslatedText"), text, **{f"{{{XML}}}lang":lang})

# ---------- OID 规则 ----------
IGoid = lambda dom: f"IG.{dom}"
IToid = lambda dom,v: f"IT.{dom}.{v}"
VLoid = lambda dom,v: f"VL.{dom}.{v}"
WCoid = lambda dom,v,i: f"WC.{dom}.{v}.V{i}"
VITo  = lambda dom,v,i: f"IT.{dom}.{v}.V{i}"
CLoid = lambda k: f"CL.{k}"; MToid=lambda k: f"MT.{k}"; COMoid=lambda k: f"COM.{k}"

# ---------- 读取 YAML ----------
def load(p): return yaml.safe_load(Path(p).read_text(encoding="utf-8"))
study     = load(STUDY_YAML)
# 从 study.yaml standards 自动推导，无需硬编码
STD_IG = next((s["oid"] for s in study.get("standards", []) if s.get("type") == "IG"), "SDTM-IG")
STD_CT = next((s["oid"] for s in study.get("standards", []) if s.get("type") == "CT"), "SDTM-CT")
codelists = load(CODELISTS) or {}
methods   = load(METHODS)   or {}
comments  = load(COMMENTS)  or {}

# 数据集结构顺序（参考 generate_define_v3.py）：主域按 Class 规范顺序（Class 内保持 TOC 行序），SUPP 统一置于所有主域之后
CLASS_ORDER_EN = ["TRIAL DESIGN", "SPECIAL-PURPOSE", "INTERVENTIONS", "EVENTS",
                  "FINDINGS", "FINDINGS ABOUT", "RELATIONSHIP", "STUDY REFERENCE"]
CLASS_ORDER_ZH = {"试验设计": "TRIAL DESIGN", "干预类": "INTERVENTIONS", "事件类": "EVENTS",
                  "发现类": "FINDINGS", "发现关于": "FINDINGS ABOUT", "关联数据": "RELATIONSHIP",
                  "研究参考": "STUDY REFERENCE"}   # SPECIAL-PURPOSE 中英一致，无需映射
def _normalize_class(v):
    if v is None: return ""
    t = str(v).strip()
    return CLASS_ORDER_ZH.get(t, t).upper()
def _class_rank(v):
    try: return CLASS_ORDER_EN.index(_normalize_class(v))
    except ValueError: return len(CLASS_ORDER_EN)

# 按 toc.yaml 加载域 yaml（排除 toc.yaml 本身）
toc_entries = load(TOC) or []                        # [{dataset, Class, ...}, ...]
toc_loaded  = [(i, e["dataset"], e.get("Class")) for i, e in enumerate(toc_entries)
               if isinstance(e, dict) and e.get("dataset")]
toc_loaded.sort(key=lambda it: (_class_rank(it[2]), it[0]))   # 先 Class 规范序，再 TOC 行序
toc_order   = [dom for _, dom, _ in toc_loaded]

ds_blocks = []; _skip_toc = []; supp_blocks = []     # (block, is_supp)
for dom in toc_order:
    fp = DS_DIR / f"{dom.lower()}.yaml"
    if not fp.exists():
        _skip_toc.append(dom); continue
    blk = load(fp)
    ds_blocks.append((blk, False))
    if blk.get("supplemental"):
        supp_blocks.append((blk["supplemental"], True))
ds_blocks.extend(supp_blocks)                        # SUPP 统一置于所有主域之后

# 收集容器（按 Define 规范顺序写出）
IGs, ITs, VLs, WCs = [], [], [], []

# ---------- 处理一个数据集块 ----------
def build_origin(parent, org):
    if not org: return
    el=SE(parent, d("Origin"), Type=org.get("type"), Source=org.get("source"))
    if org.get("description"): TT(el, org["description"])
    for pg in org.get("crf_pages", []):
        dr=SE(el, d("DocumentRef"), leafID="Annotated CRF")
        SE(dr, d("PDFPageRef"), PageRefs=pg, Type="PhysicalRef")
    for doc in org.get("documents", []):
        SE(el, d("DocumentRef"), leafID=doc)

def build_itemdef(oid, name, vy, *, codelist=None, valuelist=None, label=None):
    it=ET.Element(o("ItemDef"), {"OID":oid, "Name":name,
                                 "DataType":str(vy.get("type")), "Length":str(vy.get("length")),
                                 "SASFieldName":name})
    if vy.get("significant_digits"): it.set("SignificantDigits", str(vy["significant_digits"]))
    if vy.get("display_format"):     it.set(d("DisplayFormat"), str(vy["display_format"]))
    if vy.get("comment"):            it.set(d("CommentOID"), COMoid(vy["comment"]))
    TT(it, label if label is not None else vy.get("label"))
    if codelist: SE(it, o("CodeListRef"), CodeListOID=CLoid(codelist))
    build_origin(it, vy.get("origin"))
    if valuelist: SE(it, d("ValueListRef"), ValueListOID=valuelist)
    ITs.append(it)
    return it

def process(block, is_supp):
    dom=block["domain"]
    ig=ET.Element(o("ItemGroupDef"), {
        "OID":IGoid(dom), "Domain":block.get("domain"), "Name":dom,
        "Repeating":"Yes" if block.get("repeating") else "No",
        "IsReferenceData":"Yes" if block.get("is_reference_data") else "No",
        "SASDatasetName":block.get("sas_dataset") or dom, "Purpose":block.get("purpose") or ("列表" if LANG=="CN" else "Tabulation"),
        d("Structure"):block.get("structure") or "", d("StandardOID"):STD_IG,
        d("ArchiveLocationID"):f"LF.{dom}"})
    if block.get("has_no_data"): ig.set(d("HasNoData"), "Yes")   # xpt 行数=0 时设定
    if block.get("comment"): ig.set(d("CommentOID"), COMoid(block["comment"]))
    TT(ig, block.get("label"))
    for vname, vy in (block.get("variables") or {}).items():
        vloid=None
        if vy.get("value_list"):
            vloid=VLoid(dom, vname)
            vl=ET.Element(d("ValueListDef"), {"OID":vloid})
            for i, ent in enumerate(vy["value_list"], 1):
                vit_oid=VITo(dom, vname, i)
                vchild={"type":vy.get("type"), "length":vy.get("length")}
                build_itemdef(vit_oid, vname, vchild,
                              codelist=ent.get("codelist"),
                              label=ent.get("label"))   # 值级 item 不再回挂 ValueListRef(避免 SUPP VLM 递归重复渲染)
                vit=ITs[-1]
                if ent.get("origin"): build_origin(vit, ent["origin"])
                if ent.get("comment"): vit.set(d("CommentOID"), COMoid(ent["comment"]))
                ref=SE(vl, o("ItemRef"), ItemOID=vit_oid, OrderNumber=i,
                       Mandatory="Yes" if ent.get("mandatory") else "No",
                       MethodOID=MToid(ent["method"]) if ent.get("method") else None)
                wcoid=WCoid(dom, vname, i)
                SE(ref, d("WhereClauseRef"), WhereClauseOID=wcoid)
                wc=ET.Element(d("WhereClauseDef"), {"OID":wcoid})
                for cond in ent.get("where", []):
                    rc=SE(wc, o("RangeCheck"), SoftHard="Soft",
                          **{d("ItemOID"):IToid(dom, cond["variable"]), "Comparator":cond.get("comparator")})
                    for val in cond.get("values", []):
                        SE(rc, o("CheckValue"), val)
                WCs.append(wc)
            VLs.append(vl)
            build_itemdef(IToid(dom, vname), vname, vy, valuelist=vloid)
        else:
            build_itemdef(IToid(dom, vname), vname, vy, codelist=vy.get("codelist"))
        SE(ig, o("ItemRef"), ItemOID=IToid(dom, vname), OrderNumber=vy.get("order"),
           Mandatory="Yes" if vy.get("mandatory") else "No",
           KeySequence=vy.get("key_sequence"),
           MethodOID=MToid(vy["method"]) if vy.get("method") else None)
    SE(ig, d("Class"), Name=block.get("class"))
    lf=SE(ig, d("leaf"), ID=f"LF.{dom}", **{f"{{{XLINK}}}href":block.get("archive") or f"{dom.lower()}.xpt"})
    SE(lf, d("title"), block.get("archive") or f"{dom.lower()}.xpt")
    IGs.append(ig)

for blk, supp in ds_blocks: process(blk, supp)

# ---------- 组装 ODM 骨架 ----------
root=ET.Element(o("ODM"), {"ODMVersion":study["file"].get("odm_version","1.3.2"),
    "FileOID":study["study"]["oid"], "FileType":"Snapshot",
    "Originator":study["file"].get("originator"), d("Context"):study["file"].get("context","Submission")})
st=SE(root, o("Study"), OID=study["study"]["oid"])
gv=SE(st, o("GlobalVariables"))
SE(gv, o("StudyName"), study["study"]["name"])
SE(gv, o("StudyDescription"), study["study"]["description"])
SE(gv, o("ProtocolName"), study["study"]["protocol"])
mv=SE(st, o("MetaDataVersion"), OID=study["metadata_version"]["oid"],
      Name=study["metadata_version"]["name"], Description=study["metadata_version"].get("description"),
      **{d("DefineVersion"):study["metadata_version"]["define_version"]})

# 1) Standards  2) AnnotatedCRF  3) SupplementalDoc
stds=SE(mv, d("Standards"))
for s in study.get("standards", []):
    SE(stds, d("Standard"), OID=s["oid"], Name=s["name"], Type=s["type"],
       Version=s["version"], Status=s.get("status"), PublishingSet=s.get("publishing_set"))
docs=study.get("documents", {})
if docs.get("annotated_crf"):
    SE(SE(mv, d("AnnotatedCRF")), d("DocumentRef"), leafID=docs["annotated_crf"]["leaf_id"])
if docs.get("supplemental"):
    sd=SE(mv, d("SupplementalDoc"))
    for x in docs["supplemental"]: SE(sd, d("DocumentRef"), leafID=x["leaf_id"])

# 4) ValueListDef  5) WhereClauseDef  6) ItemGroupDef  7) ItemDef
for e in VLs: mv.append(e)
for e in WCs: mv.append(e)
for e in IGs: mv.append(e)
for e in ITs: mv.append(e)

# 8) CodeList
for key, cl in codelists.items():
    el=SE(mv, o("CodeList"), OID=CLoid(key), Name=cl.get("name"), DataType=cl.get("data_type","text"))
    el.set(d("StandardOID"), cl.get("standard", STD_CT))
    if cl.get("type")=="external":
        SE(el, o("ExternalCodeList"), Dictionary=cl["dictionary"], Version=cl["version"], href=cl.get("href"))
    else:
        itemtag = "CodeListItem" if cl.get("type")=="decoded" else "EnumeratedItem"
        for it in cl.get("items", []):
            ci=SE(el, o(itemtag), CodedValue=it["value"])
            if it.get("extended"): ci.set(d("ExtendedValue"),"Yes")
            if "decode" in it:
                dec=SE(ci, o("Decode")); SE(dec, o("TranslatedText"), it["decode"], **{f"{{{XML}}}lang":XML_LANG})
            if it.get("nci_code"): SE(ci, o("Alias"), Name=it["nci_code"], Context="nci:ExtCodeID")
    if cl.get("nci_code"): SE(el, o("Alias"), Name=cl["nci_code"], Context="nci:ExtCodeID")

# 9) MethodDef
for key, m in methods.items():
    me=SE(mv, o("MethodDef"), OID=MToid(key), Name=m.get("name"), Type=m.get("type","Computation"))
    TT(me, m.get("description"))

# 10) CommentDef
for key, cm in comments.items():
    ce=SE(mv, d("CommentDef"), OID=COMoid(key))
    if isinstance(cm, dict):
        TT(ce, cm.get("description"))
        if cm.get("doc_id"):
            dr=SE(ce, d("DocumentRef"), leafID=cm["doc_id"])
            if cm.get("page_refs"):
                SE(dr, d("PDFPageRef"), PageRefs=str(cm["page_refs"]), Type=cm.get("page_ref_type","PhysicalRef"))
    else:
        TT(ce, cm)

# 11) leaf
for k, meta in [("annotated_crf",docs.get("annotated_crf"))]+[("supp",x) for x in docs.get("supplemental",[])]:
    if not meta: continue
    lf=SE(mv, d("leaf"), ID=meta["leaf_id"], **{f"{{{XLINK}}}href":meta.get("href")})
    SE(lf, d("title"), meta.get("title"))

# ---------- 写出（带 xsl 处理指令 + 缩进） ----------
ET.indent(root, space="  ")
xml_body=ET.tostring(root, encoding="unicode")
header='<?xml version="1.0" encoding="UTF-8"?>\n<?xml-stylesheet type="text/xsl" href="./define2-1.xsl"?>\n'
OUT_FILE.write_text(header+xml_body, encoding="utf-8")

# ---------- 往返完整性自检 + 报告 ----------
chk=ET.parse(OUT_FILE).getroot()
def cnt(tag): return len(chk.findall(".//"+tag))

defined=set(e.get("OID") for e in chk.iter() if e.get("OID"))
REF_ATTRS={"ItemOID","CodeListOID","ValueListOID","WhereClauseOID","MethodOID","CommentOID","leafID"}
leaf_ids=set(e.get("ID") for e in chk.iter() if e.tag==d("leaf"))
broken=[]
for e in chk.iter():
    for a,v in e.attrib.items():
        an=a.split('}')[-1]
        if an=="leafID":
            if v not in leaf_ids: broken.append((an,v))
        elif an in REF_ATTRS and v not in defined:
            broken.append((an,v))

def _sz(p):
    b=os.path.getsize(p); return f"{b/1e6:.1f} MB" if b>=1e6 else f"{b/1e3:.1f} KB"
_BAR, _SUB = "═"*58, "─"*58
_counts=[("ItemGroupDef",o("ItemGroupDef")),("ItemDef",o("ItemDef")),("CodeList",o("CodeList")),
         ("MethodDef",o("MethodDef")),("CommentDef",d("CommentDef")),
         ("ValueListDef",d("ValueListDef")),("WhereClauseDef",d("WhereClauseDef"))]

print("\n"+_BAR)
print("  ✅ Define-XML 2.1 生成 (define.xml)")
print(_BAR)
print(f"  📤 {_pad('输出',6)}{OUT_FILE.resolve()}")
print(f"  📐 {_pad('大小',6)}{_sz(OUT_FILE)}")
print(_SUB)
print("  🔢 元素统计")
for lab,tag in _counts:
    note = "   (= 数据集变量 + 值水平条件)" if lab=="ItemDef" else ""
    print(f"       · {_pad(lab,16)}{cnt(tag):>5}{note}")
print(_SUB)
print("  🔗 断链引用    " + ("无 ✅" if not broken else f"{len(broken)} 处 ❌ {broken[:5]}"))
if _skip_toc:
    print(f"  ⚠ 缺域        toc 列了但 datasets/ 缺文件 {len(_skip_toc)} 个: {_skip_toc}")
print(f"  📋 数据集顺序  {toc_order}")
print(_SUB)
_cell = time.time() - _t0
try:
    _elapsed = time.time() - _notebook_start
    print(f"  ⏱  本 cell {_cell:.1f} s   ·   全流程 {_elapsed/60:.2f} min ({_elapsed:.1f} s)")
except NameError:
    print(f"  ⏱  本 cell {_cell:.1f} s   ·   (全流程计时不可用: 请从 config cell 开始 Run All)")
print(_BAR)


══════════════════════════════════════════════════════════
  ✅ Define-XML 2.1 生成 (define.xml)
══════════════════════════════════════════════════════════
  📤 输出  C:\Users\chenqx\Desktop\ai-define-master\ai-define-master\define.xml
  📐 大小  730.8 KB
──────────────────────────────────────────────────────────
  🔢 元素统计
       · ItemGroupDef       55
       · ItemDef          1081   (= 数据集变量 + 值水平条件)
       · CodeList          145
       · MethodDef          33
       · CommentDef         55
       · ValueListDef       29
       · WhereClauseDef    201
──────────────────────────────────────────────────────────
  🔗 断链引用    16 处 ❌ [('MethodOID', 'MT.EXSTDY'), ('MethodOID', 'MT.EXENDY'), ('CommentOID', 'COM.TV.ARM'), ('CommentOID', 'COM.NC'), ('CommentOID', 'COM.NC')]
  📋 数据集顺序  ['TA', 'TE', 'TI', 'TS', 'TV', 'DM', 'DC', 'SE', 'SV', 'CM', 'EC', 'EX', 'PR', 'AE', 'CE', 'DS', 'DV', 'MH', 'CV', 'DD', 'EG', 'IE', 'IS', 'LB', 'MB', 'MI', 'PC', 'PE', 'RP', 'RS', 'SS', 'TR', 'TU', 'VS']
──────────────